In [4]:
# ============================================
# BioClinicalBERT + Encrypted Last-Layer Calibration (CKKS) on MIMIC-III Notes
# One-cell end-to-end: training, calibration, multi-seed summary, per-seed summary, degree-sweep summary
# ============================================

import os, sys, re, time, platform, importlib, json, hashlib, warnings, math, glob
warnings.filterwarnings("ignore")
os.environ["TOKENIZERS_PARALLELISM"] = "false"  # avoid HF fork warnings

from typing import List, Tuple, Dict, Optional
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# -----------------------------
# 0) Reproducibility & settings
# -----------------------------
SEED = 42
np.random.seed(SEED)

def set_torch_seed(seed=SEED):
    try:
        import torch
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    except Exception:
        pass

def maybe_import(name):
    try: return importlib.import_module(name)
    except Exception: return None

# Soft deps
sklearn = maybe_import("sklearn")
tfm     = maybe_import("transformers")
torch_m = maybe_import("torch")
ts      = maybe_import("tenseal")

VERSIONS = {}
for pkg in ["transformers","torch","tenseal","sklearn","pandas","numpy","matplotlib"]:
    try:
        m = importlib.import_module(pkg)
        VERSIONS[pkg] = getattr(m, "__version__", "unknown")
    except Exception:
        VERSIONS[pkg] = "not installed"

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
for k,v in VERSIONS.items(): print(f"{k}: {v}")

# -----------------------------
# 1) Paths, toggles, parameters
# -----------------------------
DATA_PATH = "final_cohort_exact_email.csv.gz"  # expects HOURS_BEFORE_DEATH + NOTES_24HR or TEXT
OUT_DIR   = "q1_ckks_outputs"
FIG_DIR   = os.path.join(OUT_DIR, "figs")
os.makedirs(OUT_DIR, exist_ok=True); os.makedirs(FIG_DIR, exist_ok=True)

# Embedding strategy
USE_TFIDF_BASELINE   = False         # True → main model uses TF-IDF instead of BERT-emb
RUN_TFIDF_BASELINE_LR= True          # LR(TF-IDF) baseline
RUN_BERT_BASELINE_LR = True          # LR(BERT-emb) baseline

TFIDF_DIM    = 1024
BERT_MODEL   = "emilyalsentzer/Bio_ClinicalBERT"
MAX_DOCS     = 50_000               # dev cap; increase for final runs
MAX_TOKENS   = 256
BATCH_EMB    = 64

# Partition / training
VAL_CAP            = 10_000        # shared, balanced validation
PRIMARY_RATIO      = 0.85          # label-skew per site
EPOCHS             = 20
BATCH_TRAIN        = 512
LR                 = 1e-3
EARLY_STOP_PATIENCE= 3

# Head / Chebyshev
HIDDEN_DIM         = 64
CHEB_DEGREES       = [2,3,4,6]
DEFAULT_CHEB_COEFFS= {
    2: [0.5, 0.3, -0.1],
    3: [0.5, 0.25, -0.12, 0.05],
    4: [0.5, 0.25, -0.12, 0.05, -0.01],
    6: [0.5, 0.3, -0.2, 0.1, -0.05, 0.02, -0.005],
}
CHEB_COEFFS        = DEFAULT_CHEB_COEFFS[3]  # default
CALIB_STEPS     = 10         # was 20
CALIB_LR        = 0.002     # was 0.1
K_CALIB         = 60        # was 100            # # of misclassified B_calib samples to build the single calibrated head
CALIB_L2        = 1e-4      # new: mild L2 to stabilize W updates

# === Calibration policy switches ===
CALIB_BIAS_ONLY = False   # False -> update W and b  (NOT bias-only)
                          # True  -> update b only   (bias-only)
CALIB_POLICY    = "BiasOnly" if CALIB_BIAS_ONLY else "FullWb"


# Multi-seed runs for mean±std reporting
RUN_SEEDS          = [42,43,44,45,46]  # Q1 tables expect ≥5 seeds

# Misc
PLOT = True  # set False to skip plotting in headless environments

# CKKS params (log for reproducibility)
CKKS_PARAMS = dict(poly_modulus_degree=8192, coeff_mod_bit_sizes=[60,40,40,60], global_scale=2**40)

# -----------------------------
# 2) Load & clean MIMIC-III CSV
# -----------------------------
if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(f"{DATA_PATH} missing")

df_raw = pd.read_csv(DATA_PATH, compression="gzip", low_memory=False)

note_col = "NOTES_24HR" if "NOTES_24HR" in df_raw.columns else ("TEXT" if "TEXT" in df_raw.columns else None)
if note_col is None:
    raise ValueError("Missing notes column (NOTES_24HR or TEXT)")
if "HOURS_BEFORE_DEATH" not in df_raw.columns:
    raise ValueError("Missing HOURS_BEFORE_DEATH")

df = df_raw.copy()
df["survival_2day"] = np.where(df["HOURS_BEFORE_DEATH"].isna() | (df["HOURS_BEFORE_DEATH"] > 48), 1, 0)

def clean_text(txt: str) -> str:
    txt = str(txt).lower()
    txt = re.sub(r"\[.*?\]", " ", txt)
    txt = re.sub(r"[^a-z0-9\s]", " ", txt)
    return re.sub(r"\s+", " ", txt).strip()

df = df.dropna(subset=[note_col, "survival_2day"])
df["NOTES"] = df[note_col].astype(str).apply(clean_text)

labels_full = df["survival_2day"].astype(np.int64).values
texts_full  = df["NOTES"].tolist()
print("Dataset size:", len(texts_full))

# Subsample for dev speed
texts  = texts_full[:MAX_DOCS]
labels = labels_full[:MAX_DOCS]

# -----------------------------
# 3) Build/cached embeddings
# -----------------------------
def sha_short(s: str) -> str:
    return hashlib.sha256(s.encode()).hexdigest()[:10]

if USE_TFIDF_BASELINE:
    if sklearn is None: raise RuntimeError("scikit-learn not available for TF-IDF baseline")
    from sklearn.feature_extraction.text import TfidfVectorizer
    vec_tag = f"tfidf_{TFIDF_DIM}"
    emb_cache = os.path.join(OUT_DIR, f"{vec_tag}_{MAX_DOCS}docs.npy")
    if os.path.exists(emb_cache):
        X_all = np.load(emb_cache, mmap_mode="r")
    else:
        VEC = TfidfVectorizer(max_features=TFIDF_DIM, ngram_range=(1,2), min_df=5, max_df=0.95, sublinear_tf=True)
        X_all = VEC.fit_transform(texts).astype(np.float32).toarray()
        np.save(emb_cache, X_all)
    EMBED_DIM = X_all.shape[1]
    embed_manifest = dict(kind="tfidf", dim=EMBED_DIM, cache=emb_cache, sklearn=VERSIONS.get("sklearn"))
else:
    if tfm is None or torch_m is None:
        raise RuntimeError("transformers/torch not available for BERT embeddings")
    EMBED_DIM  = 768
    bert_tag = f"bioclinbert_{MAX_TOKENS}"
    emb_cache = os.path.join(OUT_DIR, f"{bert_tag}_{MAX_DOCS}docs.npy")

    def compute_llm_embeddings(texts: List[str]) -> np.ndarray:
        from transformers import AutoTokenizer, AutoModel
        device = "cuda" if torch_m.cuda.is_available() else "cpu"
        tok   = AutoTokenizer.from_pretrained(BERT_MODEL, use_fast=True)
        model = AutoModel.from_pretrained(BERT_MODEL).to(device).eval()
        feats = []
        with torch_m.no_grad():
            for i in range(0, len(texts), BATCH_EMB):
                batch = texts[i:i+BATCH_EMB]
                enc = tok(batch, truncation=True, padding=True, max_length=MAX_TOKENS, return_tensors="pt").to(device)
                out = model(**enc).last_hidden_state[:, 0, :].cpu().numpy()
                feats.append(out)
        return np.vstack(feats).astype(np.float32)

    if os.path.exists(emb_cache):
        X_all = np.load(emb_cache, mmap_mode="r")
    else:
        X_all = compute_llm_embeddings(texts)
        np.save(emb_cache, X_all)
    embed_manifest = dict(kind="bioclinicalbert", dim=EMBED_DIM, cache=emb_cache, model=BERT_MODEL, transformers=VERSIONS.get("transformers"))

print("Embeddings shape:", X_all.shape)

# -----------------------------
# 4) Utilities (metrics, plots)
# -----------------------------
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support, roc_auc_score, average_precision_score,
    confusion_matrix, roc_curve, precision_recall_curve, brier_score_loss
)

def expected_calibration_error(y_true, prob, n_bins=15):
    y_true = np.asarray(y_true).astype(int)
    prob = np.asarray(prob).astype(float)
    bins = np.linspace(0.0, 1.0, n_bins+1)
    inds = np.digitize(prob, bins) - 1
    ece = 0.0
    for b in range(n_bins):
        mask = inds == b
        if not np.any(mask): continue
        conf = prob[mask].mean()
        acc  = (y_true[mask] == (prob[mask] >= 0.5)).mean()
        ece += (np.sum(mask)/len(prob)) * abs(acc - conf)
    return float(ece)

def reliability_plot(tag, y_true, prob, fig_dir=FIG_DIR, n_bins=15):
    y_true = np.asarray(y_true).astype(int)
    prob = np.asarray(prob).astype(float)
    bins = np.linspace(0.0, 1.0, n_bins+1)
    inds = np.digitize(prob, bins) - 1
    xs, ys = [], []
    for b in range(n_bins):
        mask = inds == b
        if not np.any(mask): continue
        xs.append(prob[mask].mean())
        ys.append((y_true[mask] == (prob[mask] >= 0.5)).mean())
    if PLOT:
        plt.figure()
        plt.plot([0,1],[0,1],'--')
        plt.plot(xs, ys, marker='o')
        plt.xlabel("Confidence")
        plt.ylabel("Empirical Accuracy")
        plt.title(f"Reliability: {tag} (ECE={expected_calibration_error(y_true, prob, n_bins):.3f})")
        plt.tight_layout()
        fn = os.path.join(fig_dir, f"reliability_{tag}.png")
        plt.savefig(fn); plt.show()

def predict_probs(model, X, batch_size=1024, device="cpu"):
    model.eval()
    ds = TensorDataset(torch.tensor(X, dtype=torch.float32))
    dl = DataLoader(ds, batch_size=batch_size, shuffle=False)
    probs = []
    with torch.no_grad():
        for (xb,) in dl:
            xb = xb.to(device)
            logits, _ = model(xb)
            p = torch.softmax(logits, dim=1)[:, 1].cpu().numpy()
            probs.append(p)
    return np.concatenate(probs, axis=0)

def metrics_at(y, prob, t=0.5):
    y = np.asarray(y).astype(int); prob = np.asarray(prob).astype(float)
    yhat = (prob >= t).astype(int)
    prec, rec, f1, _ = precision_recall_fscore_support(y, yhat, average="binary", zero_division=0)
    out = dict(
        Acc=accuracy_score(y, yhat),
        Prec=prec, Rec=rec, F1=f1,
        AUROC=roc_auc_score(y, prob),
        AUPRC=average_precision_score(y, prob),
        Brier=brier_score_loss(y, prob)
    )
    cm = confusion_matrix(y, yhat).tolist()
    out["ConfMat"] = cm
    return out

def best_threshold(y, prob, metric="F1"):
    ts = np.linspace(0.05, 0.95, 19)
    best_t, best_v = 0.5, -1.0
    for t in ts:
        v = metrics_at(y, prob, t)[metric]
        if not np.isnan(v) and v > best_v:
            best_v, best_t = v, t
    return best_t, best_v

def dump_curves(tag, y_true, prob, out_dir=OUT_DIR, fig_dir=FIG_DIR):
    fpr, tpr, _ = roc_curve(y_true, prob)
    prec, rec, _ = precision_recall_curve(y_true, prob)
    pd.DataFrame({"fpr": fpr, "tpr": tpr}).to_csv(os.path.join(out_dir, f"roc_{tag}.csv"), index=False)
    pd.DataFrame({"precision": prec, "recall": rec}).to_csv(os.path.join(out_dir, f"pr_{tag}.csv"), index=False)
    if PLOT:
        plt.figure(); plt.plot(fpr, tpr)
        plt.xlabel("FPR"); plt.ylabel("TPR"); plt.title(f"ROC: {tag}")
        plt.tight_layout(); plt.savefig(os.path.join(fig_dir, f"roc_{tag}.png")); plt.show()
        plt.figure(); plt.plot(rec, prec)
        plt.xlabel("Recall"); plt.ylabel("Precision"); plt.title(f"PR: {tag}")
        plt.tight_layout(); plt.savefig(os.path.join(fig_dir, f"pr_{tag}.png")); plt.show()

# -----------------------------
# 5) Model head & training
# -----------------------------
device = "cuda" if (torch_m is not None and torch_m.cuda.is_available()) else "cpu"

class SimpleHead(nn.Module):
    def __init__(self, input_dim, hidden=HIDDEN_DIM, num_classes=2, p_drop=0.10):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, hidden)
        self.drop= nn.Dropout(p_drop)
        self.fc2 = nn.Linear(hidden, num_classes)  # logits
    def forward(self, x):
        x = torch.relu(self.fc1(x))
        feat = self.drop(x)
        return self.fc2(feat), feat  # logits, penultimate

def class_weights_from_labels(y: np.ndarray) -> torch.Tensor:
    classes, counts = np.unique(y, return_counts=True)
    total = counts.sum()
    weights = total / (len(classes) * counts.astype(np.float32))
    w = torch.ones(2, dtype=torch.float32)
    for c, wt in zip(classes, weights): w[int(c)] = float(wt)
    return w

def train_head(X_train, y_train, X_val, y_val, input_dim, epochs=EPOCHS, batch_size=BATCH_TRAIN, lr=LR, patience=EARLY_STOP_PATIENCE):
    model = SimpleHead(input_dim=input_dim).to(device)
    weight = class_weights_from_labels(y_train).to(device)
    lossfn = nn.CrossEntropyLoss(weight=weight)
    opt = optim.Adam(model.parameters(), lr=lr)

    ds_tr = TensorDataset(torch.tensor(X_train, dtype=torch.float32), torch.tensor(y_train, dtype=torch.long))
    ds_va = TensorDataset(torch.tensor(X_val,   dtype=torch.float32), torch.tensor(y_val,   dtype=torch.long))
    dl_tr = DataLoader(ds_tr, batch_size=batch_size, shuffle=True,  drop_last=False)
    dl_va = DataLoader(ds_va, batch_size=batch_size, shuffle=False, drop_last=False)

    best_val = float("inf"); best_state = None; wait = 0

    for epoch in range(epochs):
        model.train(); running = 0.0
        for xb, yb in dl_tr:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            logits, _ = model(xb)
            loss = lossfn(logits, yb)
            loss.backward(); opt.step()
            running += loss.item() * xb.size(0)
        tr_loss = running / len(ds_tr)

        model.eval(); v_running = 0.0
        with torch.no_grad():
            for xb, yb in dl_va:
                xb, yb = xb.to(device), yb.to(device)
                logits, _ = model(xb)
                v_running += lossfn(logits, yb).item() * xb.size(0)
        va_loss = v_running / len(ds_va)
        print(f"Epoch {epoch+1:02d}: train loss={tr_loss:.4f}, val loss={va_loss:.4f}")

        if va_loss + 1e-6 < best_val:
            best_val = va_loss
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            wait = 0
        else:
            wait += 1
            if wait >= patience:
                print(f"Early stopping triggered at epoch {epoch+1}.")
                break

    if best_state is not None:
        model.load_state_dict(best_state)
    return model

# -----------------------------
# 6) Chebyshev softmax (HE-friendly; clamp to fixed L from validation)
# -----------------------------
def cheb_softmax_clamped(logits_row: np.ndarray, coeffs: List[float], L: float) -> np.ndarray:
    # 1) clamp logits to [-L, L]
    z = np.clip(logits_row, -L, L)
    # 2) map to [-1, 1]
    x = z / (L + 1e-12)
    # 3) Chebyshev series
    T0 = np.ones_like(x)
    T1 = x.copy()
    T  = [T0, T1]
    for _ in range(2, len(coeffs)):
        Tn = 2 * x * T[-1] - T[-2]
        T.append(Tn)
    poly = sum(c * Tn for c, Tn in zip(coeffs, T))
    poly = np.clip(poly, 1e-6, np.inf)
    return poly / np.sum(poly)

# Backward-compat wrapper:
def cheb_softmax(logits_row: np.ndarray, coeffs: List[float], L: Optional[float] = None) -> np.ndarray:
    assert L is not None, "Provide L from validation for clamped Chebyshev softmax."
    return cheb_softmax_clamped(logits_row, coeffs, L)

# -----------------------------
# 7) One SEED run (returns dict of outputs)
# -----------------------------
from sklearn.model_selection import train_test_split

def run_one_seed(seed:int=42) -> Dict:
    set_torch_seed(seed)
    rng = np.random.default_rng(seed)

    # shared 50/50 validation (balanced)
    pos_idx, neg_idx = np.where(labels==1)[0], np.where(labels==0)[0]
    n_val = min(VAL_CAP//2, len(pos_idx), len(neg_idx))
    val_idx = np.concatenate([rng.choice(pos_idx, n_val, replace=False),
                              rng.choice(neg_idx, n_val, replace=False)])
    train_mask = np.ones(len(labels), bool); train_mask[val_idx] = False

    X_val, y_val = X_all[val_idx], labels[val_idx]
    X_pool, y_pool = X_all[train_mask], labels[train_mask]

    def skew(X, y, primary_labels, primary_ratio=PRIMARY_RATIO, seed=seed):
        r = np.random.default_rng(seed)
        mask = np.isin(y, primary_labels)
        Xp, yp = X[mask], y[mask]
        Xs, ys = X[~mask], y[~mask]
        k = int((1-primary_ratio) * len(Xp)); k = min(max(k,0), len(Xs))
        idx = r.choice(len(Xs), k, replace=False) if k>0 else np.array([], int)
        Xf = np.concatenate([Xp, Xs[idx]], 0)
        yf = np.concatenate([yp, ys[idx]], 0)
        perm = r.permutation(len(Xf))
        return Xf[perm], yf[perm]

    X_A,     y_A     = skew(X_pool, y_pool, [0], seed=seed)     # A enriched for class 0
    X_B_all, y_B_all = skew(X_pool, y_pool, [1], seed=seed+1)   # B enriched for class 1

    # First: Train / (Calib+Test)
    X_B_train, X_B_rest, y_B_train, y_B_rest = train_test_split(
        X_B_all, y_B_all, test_size=0.2, stratify=y_B_all, random_state=seed+2
    )
    # Second: split the 20% into half Calib, half Test (10%/10% overall)
    X_B_calib, X_B_test, y_B_calib, y_B_test = train_test_split(
        X_B_rest, y_B_rest, test_size=0.5, stratify=y_B_rest, random_state=seed+3
    )
    print(f"A train: {X_A.shape} | B_train: {X_B_train.shape} | B_calib: {X_B_calib.shape} | B_test: {X_B_test.shape} | Val: {X_val.shape}")

    # Train heads
    mA = train_head(X_A, y_A, X_val, y_val, input_dim=X_A.shape[1])
    mB = train_head(X_B_train, y_B_train, X_val, y_val, input_dim=X_B_train.shape[1])

    # Validation penultimate via mB (to match test path)
    with torch.no_grad():
        ds_val = TensorDataset(torch.tensor(X_val, dtype=torch.float32))
        dl_val = DataLoader(ds_val, batch_size=1024, shuffle=False)
        feats_val = []
        mB.eval()  # <<< ADD THIS
        for (xb,) in dl_val:
            xb = xb.to(device)
            _, f = mB(xb)
            feats_val.append(f.cpu().numpy())

    F_val = np.vstack(feats_val)  # (N_val, 64)

    # Fix Chebyshev clamp L on validation logits (robust to outliers)
    W_base = mA.fc2.weight.detach().cpu().numpy().T   # (64,2)
    b_base = mA.fc2.bias.detach().cpu().numpy()
    logits_val = F_val @ W_base + b_base
    L_CHEB = float(np.percentile(np.abs(logits_val), 99.0, axis=None))
    L_CHEB = max(L_CHEB, 1.0)


    # One frozen threshold t* from validation (using A-head over F_val)
    probs_val_pre = np.array([cheb_softmax(row, CHEB_COEFFS, L=L_CHEB)[1] for row in logits_val])
    t_star, _ = best_threshold(y_val, probs_val_pre, metric="F1")

    # === NEW: dump A-head logits for validation ===
    import pandas as _pd, numpy as _np
    _ahead_val_logits = (logits_val[:,1] - logits_val[:,0]).ravel()  # logit = z1 - z0
    _pd.DataFrame({"logit": _np.asarray(_ahead_val_logits), "y": y_val.astype(int)}).to_csv(
        os.path.join(OUT_DIR, "ahead_val_logits.csv"), index=False
    )


    # Baselines (optional LR on embeddings)
    baseline_rows=[]
    if sklearn is not None:
        from sklearn.linear_model import LogisticRegression

        def eval_lr(X_tr, y_tr, X_te, y_te, tag):
            lr_clf = LogisticRegression(max_iter=1000, solver="lbfgs", class_weight="balanced", n_jobs=-1)
            lr_clf.fit(X_tr, y_tr)
            prob = lr_clf.predict_proba(X_te)[:,1]
            t, _ = best_threshold(y_te, prob, metric="F1")
            yhat = (prob >= t).astype(int)
            row = dict(
                model=tag, AUROC=float(roc_auc_score(y_te, prob)),
                AUPRC=float(average_precision_score(y_te, prob)),
                F1=float(precision_recall_fscore_support(y_te, yhat, average="binary", zero_division=0)[2]),
                Acc=float(accuracy_score(y_te, yhat)), t_star=float(t)
            )
            return row

        if RUN_BERT_BASELINE_LR and embed_manifest["kind"] == "bioclinicalbert":
            baseline_rows.append(eval_lr(X_pool, y_pool, X_val, y_val, f"LR(BERT-emb)_seed{seed}"))
        if RUN_TFIDF_BASELINE_LR:
            try:
                from sklearn.feature_extraction.text import TfidfVectorizer
                VEC_BL = TfidfVectorizer(max_features=TFIDF_DIM, ngram_range=(1,2), min_df=5, max_df=0.95, sublinear_tf=True)
                X_all_tfidf = VEC_BL.fit_transform(texts).astype(np.float32).toarray()
                X_val_tf = X_all_tfidf[val_idx]
                X_pool_tf= X_all_tfidf[train_mask]
                y_pool_tf= y_pool
                baseline_rows.append(eval_lr(X_pool_tf, y_pool_tf, X_val_tf, y_val, f"LR(TF-IDF)_seed{seed}"))
            except Exception as e:
                print("[warn] TF-IDF LR baseline skipped:", e)

    # Helper to get penultimate features
    def penultimate_from_array(X_arr, idx):
        with torch.no_grad():
            mB.eval()  # <<< ADD THIS
            xb = torch.tensor(X_arr[idx].reshape(1, -1), dtype=torch.float32).to(device)
            _, feat = mB(xb)
        return feat.cpu().numpy()[0]


    # Select misclassified indices under A-head (Chebyshev + clamp L_CHEB)
    def pick_misclassified_indices(max_k=K_CALIB, coeffs=CHEB_COEFFS):
        """
        Build a class-balanced set of the hardest misclassified examples (smallest margin).
        """
        cand = {0: [], 1: []}  # store (margin, idx) pairs per class
        for i in range(len(X_B_calib)):
            f = penultimate_from_array(X_B_calib, i)
            p = cheb_softmax(f @ W_base + b_base, coeffs, L=L_CHEB)
            y_i = int(y_B_calib[i])
            y_hat = int(np.argmax(p))
            if y_hat != y_i:
                # hardness = distance from decision: smaller |p1-0.5| => harder
                margin = abs(p[1] - 0.5)
                cand[y_i].append((margin, i))
    
        # sort by hardness (ascending margin → harder)
        for c in (0, 1):
            cand[c].sort(key=lambda t: t[0])
    
        # balanced take
        per_class = max_k // 2
        sel = [idx for _, idx in cand[0][:per_class]] + [idx for _, idx in cand[1][:per_class]]
    
        # if one side runs short, fill from the other (still hardest first)
        if len(sel) < max_k:
            remaining = max_k - len(sel)
            pool = cand[0][per_class:] + cand[1][per_class:]
            pool.sort(key=lambda t: t[0])
            sel += [idx for _, idx in pool[:remaining]]
    
        return sel[:max_k]


    mis_idx = pick_misclassified_indices(K_CALIB, CHEB_COEFFS)
    print(f"Selected {len(mis_idx)} misclassified B_calib samples for calibration batch (seed={seed}).")

    # Optional HE timings + size
    he_timings = {}; he_payload_bytes=None
    if ts is not None and len(mis_idx)>0:
        try:
            f0 = penultimate_from_array(X_B_calib, mis_idx[0])
            t0 = time.time()
            context = ts.context(ts.SCHEME_TYPE.CKKS, poly_modulus_degree=CKKS_PARAMS["poly_modulus_degree"],
                                 coeff_mod_bit_sizes=CKKS_PARAMS["coeff_mod_bit_sizes"])
            context.global_scale = CKKS_PARAMS["global_scale"]
            context.generate_galois_keys()
            he_timings["context_ms"] = (time.time()-t0)*1000

            t1=time.time(); enc_feat = ts.ckks_vector(context, f0.tolist())
            he_timings["encrypt_feat_ms"]=(time.time()-t1)*1000
            try: he_payload_bytes = len(enc_feat.serialize())
            except Exception: he_payload_bytes=None

            t2=time.time(); logits_he=[]
            for j in range(W_base.shape[1]):
                lj = enc_feat.dot(W_base[:, j].tolist()).decrypt()[0] + float(b_base[j])
                logits_he.append(lj)
            he_timings["enc_dot_total_ms"]=(time.time()-t2)*1000
            print("HE warmup logits (decrypted):", np.round(logits_he,4))
        except Exception as e:
            print("[warn] HE timing step skipped:", e)

    # ------- Build a SINGLE calibrated head from the misclassified batch -------
    W_cal = W_base.copy(); b_cal = b_base.copy()
    W_base_copy = W_base.copy()
    b_base_copy = b_base.copy()

    def calibrate_once(f, ytrue, coeffs=CHEB_COEFFS, steps=CALIB_STEPS, lr=CALIB_LR, 
                   W=None, b=None, bias_only=CALIB_BIAS_ONLY):
        """
        One-sample calibration loop with mild L2 regularization to base params.
        """
        if W is None: W = W_cal
        if b is None: b = b_cal
    
        target = np.array([1.0 if k==ytrue else 0.0 for k in range(2)], dtype=float)
    
        for _ in range(steps):
            p = cheb_softmax(f @ W + b, coeffs, L=L_CHEB)  # (2,)
            err = target - p
    
            if bias_only:
                for j in range(W.shape[1]):
                    b[j] += lr * (err[j] - CALIB_L2 * (b[j] - b_base_copy[j]))
            else:
                for j in range(W.shape[1]):
                    W[:, j] += lr * (err[j] * f - CALIB_L2 * (W[:, j] - W_base_copy[:, j]))
                    b[j]    += lr * (err[j] - CALIB_L2 * (b[j] - b_base_copy[j]))
        return W, b



    t_start = time.time()
    for idx_m in mis_idx:
        f = penultimate_from_array(X_B_calib, idx_m)
        ytrue = int(y_B_calib[idx_m])
        W_cal, b_cal = calibrate_once(
            f, ytrue, CHEB_COEFFS, CALIB_STEPS, CALIB_LR, 
            W_cal, b_cal, bias_only=CALIB_BIAS_ONLY
        )
    calib_batch_ms = (time.time()-t_start)*1000

    # ------- Evaluate PRE/POST calibration on FULL B_test -------
    feats = []
    with torch.no_grad():
        ds = TensorDataset(torch.tensor(X_B_test, dtype=torch.float32))
        dl = DataLoader(ds, batch_size=1024, shuffle=False)
        mB.eval()  # <<< ADD THIS
        for (xb,) in dl:
            xb = xb.to(device)
            _, f = mB(xb)
            feats.append(f.cpu().numpy())
    F_B = np.vstack(feats)  # (N,64)


    # === NEW: dump A-head logits for B_test ===
    logits_btest_ahead = F_B @ W_base + b_base
    _ahead_btest_logits = (logits_btest_ahead[:,1] - logits_btest_ahead[:,0]).ravel()
    _pd.DataFrame({"logit": _np.asarray(_ahead_btest_logits), "y": y_B_test.astype(int)}).to_csv(
        os.path.join(OUT_DIR, "ahead_btest_logits.csv"), index=False
    )


    def probs_from_params(F, W, b, coeffs=CHEB_COEFFS):
        logits = F @ W + b
        out = np.empty((len(F),), dtype=float)
        for i in range(len(F)):
            out[i] = cheb_softmax(logits[i], coeffs, L=L_CHEB)[1]  # prob of class 1
        return out

    probs_pre  = probs_from_params(F_B, W_base, b_base, CHEB_COEFFS)
    probs_post = probs_from_params(F_B, W_cal,  b_cal,  CHEB_COEFFS)
    # --- Temperature scaling on validation, applied to post-calibration logits ---
    # Build validation logits from calibrated head
    logits_val_post = F_val @ W_cal + b_cal
    # Convert 2-class logits to a single logit = z1 - z0
    logit_val_1d = (logits_val_post[:, 1] - logits_val_post[:, 0]).ravel()
    
    def _sigmoid(z): 
        return 1.0 / (1.0 + np.exp(-z))
    
    def _fit_temperature(lv, yv):
        t_candidates = np.linspace(0.5, 5.0, 91)
        best_t, best_nll = 1.0, 1e9
        yv = yv.astype(float)
        eps = 1e-9
        for t in t_candidates:
            p = _sigmoid(lv / t)
            nll = -(yv*np.log(p+eps) + (1-yv)*np.log(1-p+eps)).mean()
            if nll < best_nll:
                best_t, best_nll = t, nll
        return best_t
    
    T_post = _fit_temperature(logit_val_1d, y_val)
    
    # Apply T to B_test logits from calibrated head
    logits_btest_post = F_B @ W_cal + b_cal
    logit_btest_1d = (logits_btest_post[:, 1] - logits_btest_post[:, 0]).ravel()
    probs_post_ts = _sigmoid(logit_btest_1d / T_post)


    # Single frozen threshold picked on validation (t_star)
    mets_pre   = metrics_at(y_B_test, probs_pre,  t_star)
    mets_post  = metrics_at(y_B_test, probs_post, t_star)
    mets_post_ts = metrics_at(y_B_test, probs_post_ts, t_star)
    
    ece_pre     = expected_calibration_error(y_B_test, probs_pre)
    ece_post    = expected_calibration_error(y_B_test, probs_post)
    ece_post_ts = expected_calibration_error(y_B_test, probs_post_ts)

    if PLOT:
        reliability_plot(f"Btest_pre_seed{seed}",  y_B_test, probs_pre)
        reliability_plot(f"Btest_post_seed{seed}", y_B_test, probs_post)
        reliability_plot(f"Btest_postTS_seed{seed}", y_B_test, probs_post_ts)  # <<< NEW

    # Persist per-seed calibration JSON
    with open(os.path.join(OUT_DIR, f"Btest_calibration_seed{seed}.json"), "w") as fjs:
        json.dump({
        "policy": CALIB_POLICY,
        "pre":      dict(threshold=float(t_star), **mets_pre,      ECE=float(ece_pre)),
        "post":     dict(threshold=float(t_star), **mets_post,     ECE=float(ece_post)),
        "post_TS":  dict(threshold=float(t_star), **mets_post_ts,  ECE=float(ece_post_ts), T=float(T_post)),
        "calibration_batch_ms": float(calib_batch_ms),
        "num_misclassified_used": int(len(mis_idx)),
        "L_CHEB": float(L_CHEB)
    }, fjs, indent=2)



    # Degree sweep (on B_test via A-head) — uses clamped Chebyshev (L_CHEB)
    def degree_sweep(degrees=CHEB_DEGREES):
        results = []
        logits = F_B @ W_base + b_base
        for d in degrees:
            coeffs = DEFAULT_CHEB_COEFFS[d]
            t0 = time.time()
            probs = np.array([cheb_softmax(row, coeffs, L=L_CHEB)[1] for row in logits])
            sweep_ms = (time.time()-t0)*1000
            t_deg, _ = best_threshold(y_B_test, probs, "F1")  # per-degree diagnostic threshold
            mets = metrics_at(y_B_test, probs, t_deg)
            results.append(dict(
                degree=d, AUROC=float(mets["AUROC"]), AUPRC=float(mets["AUPRC"]),
                F1=float(mets["F1"]), Acc=float(mets["Acc"]),
                t_star=float(t_deg), time_ms=float(sweep_ms),
                ECE=float(expected_calibration_error(y_B_test, probs))
            ))
        return pd.DataFrame(results)

    deg_df = degree_sweep(CHEB_DEGREES)
    deg_df.to_csv(os.path.join(OUT_DIR, f"chebyshev_degree_sweep_seed{seed}.csv"), index=False)
    if PLOT:
        plt.figure(); plt.plot(deg_df["degree"], deg_df["AUROC"], marker="o")
        plt.xlabel("Chebyshev degree"); plt.ylabel("AUROC")
        plt.title(f"Chebyshev Degree Sweep (seed={seed})")
        plt.tight_layout(); plt.savefig(os.path.join(FIG_DIR, f"cheb_degree_auroc_seed{seed}.png")); plt.show()

    # Validation summary payload (kept for completeness)
    valA = predict_probs(mA, X_val, device=device);  tA, _ = best_threshold(y_val, valA, metric="F1")
    valB = predict_probs(mB, X_val, device=device);  tB, _ = best_threshold(y_val, valB, metric="F1")
    mA_05 = metrics_at(y_val, valA, 0.5); mA_t = metrics_at(y_val, valA, tA)
    mB_05 = metrics_at(y_val, valB, 0.5); mB_t = metrics_at(y_val, valB, tB)

    dump_curves(f"A_val_seed{seed}", y_val, valA)
    dump_curves(f"B_val_seed{seed}", y_val, valB)
    reliability_plot(f"A_val_seed{seed}", y_val, valA)
    reliability_plot(f"B_val_seed{seed}", y_val, valB)


    val_payload = dict(
        A_at_0p5=mA_05, A_at_tstar=dict(t=float(tA), **mA_t),
        B_at_0p5=mB_05, B_at_tstar=dict(t=float(tB), **mB_t),
    )

    return dict(
        seed=seed,
        splits=dict(A=X_A.shape, Btr=X_B_train.shape, Bte=X_B_test.shape, Val=X_val.shape),
        val=val_payload,
        baselines=baseline_rows,
        he=dict(params=CKKS_PARAMS, timings_ms=he_timings, payload_bytes=he_payload_bytes),
                btest_calibration=dict(
            pre=dict(t=float(t_star),  **mets_pre,  ECE=float(ece_pre)),
            post=dict(t=float(t_star), **mets_post, ECE=float(ece_post)),
            post_TS=dict(t=float(t_star), **mets_post_ts, ECE=float(ece_post_ts), T=float(T_post)),  # <<< NEW
            batch_ms=float(calib_batch_ms),
            k_used=int(len(mis_idx)),
            L_CHEB=float(L_CHEB)
        ),
        degree_sweep_path=os.path.join(OUT_DIR, f"chebyshev_degree_sweep_seed{seed}.csv")
    )

# -----------------------------
# 8) Run multi-seed + aggregate
# -----------------------------
all_runs = []
for s in RUN_SEEDS:
    print("\n" + "="*12 + f" SEED {s} " + "="*12)
    out = run_one_seed(s)
    all_runs.append(out)

# Aggregate tables: Validation (A & B at t*), B_test pre/post
def agg_mean_std(rows: List[float]) -> Dict[str, float]:
    arr = np.array(rows, dtype=float)
    return dict(mean=float(arr.mean()), std=float(arr.std(ddof=0)))

def collect_metric(path_fn):
    vals = []
    for r in all_runs:
        try:
            v = path_fn(r)
        except KeyError:
            continue  # skip seeds missing this metric
        vals.append(v)
    return agg_mean_std(vals) if vals else dict(mean=float("nan"), std=float("nan"))


summary = {
    "seeds": [r["seed"] for r in all_runs],
    "A_val_AUROC": collect_metric(lambda r: r["val"]["A_at_tstar"]["AUROC"]),
    "A_val_AUPRC": collect_metric(lambda r: r["val"]["A_at_tstar"]["AUPRC"]),
    "A_val_F1":    collect_metric(lambda r: r["val"]["A_at_tstar"]["F1"]),
    "B_val_AUROC": collect_metric(lambda r: r["val"]["B_at_tstar"]["AUROC"]),
    "B_val_AUPRC": collect_metric(lambda r: r["val"]["B_at_tstar"]["AUPRC"]),
    "B_val_F1":    collect_metric(lambda r: r["val"]["B_at_tstar"]["F1"]),
    "Btest_pre_AUROC":  collect_metric(lambda r: r["btest_calibration"]["pre"]["AUROC"]),
    "Btest_pre_AUPRC":  collect_metric(lambda r: r["btest_calibration"]["pre"]["AUPRC"]),
    "Btest_pre_F1":     collect_metric(lambda r: r["btest_calibration"]["pre"]["F1"]),
    "Btest_pre_Brier":  collect_metric(lambda r: r["btest_calibration"]["pre"]["Brier"]),
    "Btest_pre_ECE":    collect_metric(lambda r: r["btest_calibration"]["pre"]["ECE"]),
    "Btest_post_AUROC": collect_metric(lambda r: r["btest_calibration"]["post"]["AUROC"]),
    "Btest_post_AUPRC": collect_metric(lambda r: r["btest_calibration"]["post"]["AUPRC"]),
    "Btest_post_F1":    collect_metric(lambda r: r["btest_calibration"]["post"]["F1"]),
    "Btest_post_Brier": collect_metric(lambda r: r["btest_calibration"]["post"]["Brier"]),
    "Btest_post_ECE":   collect_metric(lambda r: r["btest_calibration"]["post"]["ECE"]),
    "Calib_batch_ms":   collect_metric(lambda r: r["btest_calibration"]["batch_ms"]),
    "Btest_postTS_AUROC": collect_metric(lambda r: r["btest_calibration"]["post_TS"]["AUROC"]),
    "Btest_postTS_AUPRC": collect_metric(lambda r: r["btest_calibration"]["post_TS"]["AUPRC"]),
    "Btest_postTS_F1":    collect_metric(lambda r: r["btest_calibration"]["post_TS"]["F1"]),
    "Btest_postTS_Brier": collect_metric(lambda r: r["btest_calibration"]["post_TS"]["Brier"]),
    "Btest_postTS_ECE":   collect_metric(lambda r: r["btest_calibration"]["post_TS"]["ECE"]),

}

with open(os.path.join(OUT_DIR, "multi_seed_summary.json"), "w") as f:
    json.dump(summary, f, indent=2)

# Save per-seed baselines table
baseline_rows_all=[]
for r in all_runs:
    for b in r["baselines"]:
        baseline_rows_all.append(dict(seed=r["seed"], **b))
if baseline_rows_all:
    pd.DataFrame(baseline_rows_all).to_csv(os.path.join(OUT_DIR, "baseline_table_all_seeds.csv"), index=False)

# -----------------------------
# 9) Provenance & manifest dump
# -----------------------------
manifest = {
    "seeds": RUN_SEEDS,
    "platform": platform.platform(),
    "python": sys.version.split()[0],
    "versions": VERSIONS,
    "data_path": os.path.abspath(DATA_PATH),
    "n_docs_used": int(len(texts)),
    "embed": embed_manifest,
    "head": {"hidden_dim": HIDDEN_DIM, "dropout": 0.10, "num_classes": 2},
    "train": {"epochs": EPOCHS, "batch_size": BATCH_TRAIN, "lr": LR, "early_stop_patience": EARLY_STOP_PATIENCE},
    "partition": {"VAL_CAP": VAL_CAP, "PRIMARY_RATIO": PRIMARY_RATIO},
    "chebyshev": {"default_coeffs": CHEB_COEFFS, "degree_sweep": CHEB_DEGREES},
    "calibration": {"steps": CALIB_STEPS, "lr": CALIB_LR, "k_batch": K_CALIB, 
                "bias_only": bool(CALIB_BIAS_ONLY), "policy": CALIB_POLICY},
    "ckks_params_logged": CKKS_PARAMS,
    "outputs": {
        "multi_seed_summary": os.path.abspath(os.path.join(OUT_DIR, "multi_seed_summary.json")),
        "baseline_table_all_seeds": os.path.abspath(os.path.join(OUT_DIR, "baseline_table_all_seeds.csv")),
        "figs_dir": os.path.abspath(FIG_DIR)
    }
}
with open(os.path.join(OUT_DIR, "manifest.json"), "w") as f:
    json.dump(manifest, f, indent=2)

# ==========================
# 10) SUMMARIZERS (Code 2 + extras) — runs automatically
# ==========================
ROOT = OUT_DIR
JSON_PATH = os.path.join(ROOT, "multi_seed_summary.json")
OUT_LONG   = os.path.join(ROOT, "summary_long.csv")    # metric,mean,std
OUT_PRETTY = os.path.join(ROOT, "summary_pretty.csv")  # metric,mean_pm_std (formatted)

# ---- Code 2: summarize multi_seed_summary.json ----
with open(JSON_PATH, "r") as f:
    js = json.load(f)

rows = []
for k, v in js.items():
    if isinstance(v, dict) and {"mean","std"}.issubset(v):
        rows.append({"metric": k, "mean": v["mean"], "std": v["std"]})

df = pd.DataFrame(rows).sort_values("metric")

rename_map = {
  "A_val_AUROC":"Val (A) AUROC","A_val_AUPRC":"Val (A) AUPRC","A_val_F1":"Val (A) F1",
  "B_val_AUROC":"Val (B) AUROC","B_val_AUPRC":"Val (B) AUPRC","B_val_F1":"Val (B) F1",
  "Btest_pre_AUROC":"B_test Pre AUROC","Btest_pre_AUPRC":"B_test Pre AUPRC",
  "Btest_pre_F1":"B_test Pre F1","Btest_pre_Brier":"B_test Pre Brier","Btest_pre_ECE":"B_test Pre ECE",
  "Btest_post_AUROC":"B_test Post AUROC","Btest_post_AUPRC":"B_test Post AUPRC",
  "Btest_post_F1":"B_test Post F1","Btest_post_Brier":"B_test Post Brier","Btest_post_ECE":"B_test Post ECE",
  "Calib_batch_ms":"Calibration Batch Time (ms)",
  "Btest_postTS_AUROC":"B_test Post+TS AUROC",
  "Btest_postTS_AUPRC":"B_test Post+TS AUPRC",
  "Btest_postTS_F1":"B_test Post+TS F1",
  "Btest_postTS_Brier":"B_test Post+TS Brier",
  "Btest_postTS_ECE":"B_test Post+TS ECE",

}
df["metric"] = df["metric"].replace(rename_map)

# Save tidy (numeric) CSV
df.to_csv(OUT_LONG, index=False)

# Save pretty CSV with "mean ± std" text
def fmt(m, s, p=3): return f"{m:.{p}f} ± {s:.{p}f}"
df_pretty = df.assign(mean_pm_std=[fmt(m,s,3) for m,s in zip(df["mean"], df["std"])])[["metric","mean_pm_std"]]
df_pretty.to_csv(OUT_PRETTY, index=False)

# ---- Extras: per-seed calibration and degree sweep summaries ----
# 1) Per-seed calibration JSONs -> one table
rows = []
for path in sorted(glob.glob(os.path.join(ROOT, "Btest_calibration_seed*.json"))):
    try:
        seed = int(path.split("seed")[-1].split(".json")[0])
    except Exception:
        continue
    with open(path, "r") as f:
        js2 = json.load(f)
    pre  = js2.get("pre", {})
    post = js2.get("post", {})
    post_ts = js2.get("post_TS", {})

    rows.append({
        "seed": seed,
        "threshold_t*": pre.get("threshold", None),
        "pre_AUROC":  pre.get("AUROC", None),
        "pre_AUPRC":  pre.get("AUPRC", None),
        "pre_F1":     pre.get("F1", None),
        "pre_Brier":  pre.get("Brier", None),
        "pre_ECE":    pre.get("ECE", None),
        "post_AUROC": post.get("AUROC", None),
        "post_AUPRC": post.get("AUPRC", None),
        "post_F1":    post.get("F1", None),
        "post_Brier": post.get("Brier", None),
        "post_ECE":   post.get("ECE", None),
        "calib_batch_ms": js2.get("calibration_batch_ms", None),
        "k_used": js2.get("num_misclassified_used", js2.get("k_used", None)),
        "L_CHEB": js2.get("L_CHEB", None),
        "postTS_AUROC": post_ts.get("AUROC", None),
        "postTS_AUPRC": post_ts.get("AUPRC", None),
        "postTS_F1":    post_ts.get("F1", None),
        "postTS_Brier": post_ts.get("Brier", None),
        "postTS_ECE":   post_ts.get("ECE", None),
        "T_post":       post_ts.get("T", None),

    })

if rows:
    df_cal = pd.DataFrame(rows).sort_values("seed")
    df_cal.to_csv(os.path.join(ROOT, "per_seed_calibration_summary.csv"), index=False)

# 2) Degree sweep CSVs -> long table + mean±std
sweep_rows = []
for path in sorted(glob.glob(os.path.join(ROOT, "chebyshev_degree_sweep_seed*.csv"))):
    try:
        seed = int(path.split("seed")[-1].split(".csv")[0])
    except Exception:
        continue
    df_sw = pd.read_csv(path)
    df_sw["seed"] = seed
    sweep_rows.append(df_sw)

if sweep_rows:
    df_sweep_long = pd.concat(sweep_rows, ignore_index=True)
    df_sweep_long.to_csv(os.path.join(ROOT, "degree_sweep_long.csv"), index=False)

    agg = (df_sweep_long
           .groupby("degree")
           .agg(AUROC_mean=("AUROC","mean"), AUROC_std=("AUROC","std"),
                AUPRC_mean=("AUPRC","mean"), AUPRC_std=("AUPRC","std"),
                F1_mean=("F1","mean"),       F1_std=("F1","std"),
                Acc_mean=("Acc","mean"),     Acc_std=("Acc","std"),
                ECE_mean=("ECE","mean"),     ECE_std=("ECE","std"),
                time_ms_mean=("time_ms","mean"), time_ms_std=("time_ms","std"))
           .reset_index())
    agg.to_csv(os.path.join(ROOT, "degree_sweep_aggregate.csv"), index=False)

print("\n✅ Done. Artifacts written to:", os.path.abspath(OUT_DIR))
print("   - multi_seed_summary.json")
print("   - summary_long.csv / summary_pretty.csv")
print("   - per_seed_calibration_summary.csv (per-seed)")
print("   - degree_sweep_long.csv / degree_sweep_aggregate.csv (if degree-sweep ran)")


########################

# ================================================
# Merged post-processing: policy table + calibration baselines
# Produces:
#   - data/mis_selection_policy.csv
#   - data/calibration_baselines.csv   (only if logits files are present)
# ================================================
import os, glob, json
import numpy as np
import pandas as pd

# --------------------------
# Setup
# --------------------------
OUT_DIR = "q1_ckks_outputs"
DATA_DIR = "data"
os.makedirs(DATA_DIR, exist_ok=True)

# --------------------------
# Part A: Aggregate "policy" results from per-seed JSONs
# --------------------------
rows = []

# 1) Primary source: policy-tagged JSONs (optional; present only if you added policy logging)
for path in glob.glob(os.path.join(OUT_DIR, "*_policy_seed*.json")):
    with open(path) as f:
        j = json.load(f)
    post = j.get("post", {})
    pol  = j.get("policy", None)
    if post and pol:
        rows.append({
            "policy": pol,
            "F1": post.get("F1"),
            "ECE": post.get("ECE"),
            "Brier": post.get("Brier"),
        })

# 2) Fallback: use your existing per-seed calibration JSONs and treat as a single policy
if not rows:
    for path in glob.glob(os.path.join(OUT_DIR, "Btest_calibration_seed*.json")):
        with open(path) as f:
            j = json.load(f)
        # new format
        post = j.get("post", {})
        # older nested format
        if not post:
            post = j.get("btest_calibration", {}).get("post", {})
        if post:
            rows.append({
                "policy": "SingleBatch",
                "F1": post.get("F1"),
                "ECE": post.get("ECE"),
                "Brier": post.get("Brier"),
            })

# Write policy table
if not rows:
    pd.DataFrame([], columns=["policy","F1","ECE","Brier"]).to_csv(
        os.path.join(DATA_DIR, "mis_selection_policy.csv"), index=False
    )
    print("[warn] No calibration JSONs found. Wrote empty data/mis_selection_policy.csv")
else:
    df = pd.DataFrame(rows)
    agg = df.groupby("policy").agg(["mean","std"])
    agg.columns = ['_'.join(col) for col in agg.columns]
    agg = agg.reset_index()

    def fmt(m, s): 
        return f"{(m if m is not None else np.nan):.3f}±{(s if s is not None else np.nan):.3f}"

    out = pd.DataFrame({
        "policy": agg["policy"],
        "F1":    [fmt(m,s) for m,s in zip(agg.get("F1_mean", pd.Series(dtype=float)),
                                          agg.get("F1_std",  pd.Series(dtype=float)))],
        "ECE":   [fmt(m,s) for m,s in zip(agg.get("ECE_mean", pd.Series(dtype=float)),
                                          agg.get("ECE_std",  pd.Series(dtype=float)))],
        "Brier": [fmt(m,s) for m,s in zip(agg.get("Brier_mean", pd.Series(dtype=float)),
                                          agg.get("Brier_std",  pd.Series(dtype=float)))],
    })
    out.to_csv(os.path.join(DATA_DIR, "mis_selection_policy.csv"), index=False)
    print("Wrote", os.path.join(DATA_DIR, "mis_selection_policy.csv"))

# --------------------------
# Part B: Calibration baselines (TS / Platt / Isotonic)
# Requires logits CSVs; will skip gracefully if missing
# --------------------------
# Expected CSVs (produce these from Code 1; see my earlier patch):
#   - q1_ckks_outputs/ahead_val_logits.csv    # columns: logit, y
#   - q1_ckks_outputs/ahead_btest_logits.csv  # columns: logit, y
# Optionally also:
#   - q1_ckks_outputs/val_lr_tfidf.csv,  test_lr_tfidf.csv
#   - q1_ckks_outputs/val_lr_bert.csv,    test_lr_bert.csv

# helpers
def sigmoid(x):
    # use numpy; no external deps needed
    return 1.0 / (1.0 + np.exp(-x))

def temperature_scale_fit(logits_val, y_val):
    # Simple line search for binary temperature scaling
    t_candidates = np.linspace(0.5, 5.0, 91)
    best_t, best_nll = 1.0, 1e9
    eps = 1e-9
    p_true = y_val.astype(float)
    for t in t_candidates:
        p = sigmoid(logits_val / t)
        nll = -(p_true*np.log(p+eps) + (1-p_true)*np.log(1-p+eps)).mean()
        if nll < best_nll:
            best_t, best_nll = t, nll
    return best_t

def ece_uniform_bins(y_true, p, n_bins=15):
    y_true = np.asarray(y_true).astype(int)
    p = np.asarray(p).astype(float)
    bins = np.linspace(0.0, 1.0, n_bins+1)
    inds = np.digitize(p, bins) - 1
    accs, confs, weights = [], [], []
    for b in range(n_bins):
        m = inds == b
        if not np.any(m): 
            continue
        confs.append(p[m].mean())
        accs.append((y_true[m] == (p[m] >= 0.5)).mean())
        weights.append(m.mean())
    if not weights:
        return np.nan
    accs, confs, weights = np.array(accs), np.array(confs), np.array(weights)
    return float(np.sum(weights * np.abs(accs - confs)))

def metrics_block(y_true, p):
    from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, brier_score_loss
    auroc = roc_auc_score(y_true, p)
    auprc = average_precision_score(y_true, p)
    f1    = f1_score(y_true, (p >= 0.5).astype(int))
    ece   = ece_uniform_bins(y_true, p, n_bins=15)
    brier = brier_score_loss(y_true, p)
    return dict(AUROC=auroc, AUPRC=auprc, F1=f1, ECE=ece, Brier=brier)

def run_calibration_set(name, val_path, test_path):
    """Return rows for TS / Platt / Isotonic given val/test CSVs with 'logit','y'."""
    try:
        from sklearn.linear_model import LogisticRegression
        from sklearn.isotonic import IsotonicRegression
    except Exception as e:
        print(f"[warn] Skipping {name} calibrations; sklearn not available: {e}")
        return []

    if not (os.path.exists(val_path) and os.path.exists(test_path)):
        print(f"[warn] Skipping {name}: missing {val_path} or {test_path}")
        return []

    v = pd.read_csv(val_path)
    t = pd.read_csv(test_path)
    lv, yv = v["logit"].values, v["y"].values.astype(int)
    lt, yt = t["logit"].values, t["y"].values.astype(int)

    out = []

    # Temperature Scaling (TS)
    T = temperature_scale_fit(lv, yv)
    p_ts = sigmoid(lt / T)
    m_ts = metrics_block(yt, p_ts)
    out.append(dict(model=f"{name}+TS", **m_ts))

    # Platt scaling (LogisticRegression on logits)
    try:
        lr = LogisticRegression(solver="lbfgs", max_iter=1000)
        lr.fit(lv.reshape(-1,1), yv)
        p_platt = lr.predict_proba(lt.reshape(-1,1))[:,1]
        m_platt = metrics_block(yt, p_platt)
        out.append(dict(model=f"{name}+Platt", **m_platt))
    except Exception as e:
        print(f"[warn] Platt scaling failed for {name}: {e}")

    # Isotonic Regression
    try:
        iso = IsotonicRegression(out_of_bounds="clip")
        iso.fit(lv, yv)
        p_iso = iso.predict(lt)
        m_iso = metrics_block(yt, p_iso)
        out.append(dict(model=f"{name}+Isotonic", **m_iso))
    except Exception as e:
        print(f"[warn] Isotonic regression failed for {name}: {e}")

    return out

cal_rows = []
# A-head (export these from Code 1 per my earlier patch)
cal_rows += run_calibration_set(
    "A-head",
    os.path.join(OUT_DIR, "ahead_val_logits.csv"),
    os.path.join(OUT_DIR, "ahead_btest_logits.csv")
)

# Optional: LR(TF-IDF) and LR(BERT-emb) if you saved these logits
cal_rows += run_calibration_set(
    "LR(TF-IDF)",
    os.path.join(OUT_DIR, "val_lr_tfidf.csv"),
    os.path.join(OUT_DIR, "test_lr_tfidf.csv")
)
cal_rows += run_calibration_set(
    "LR(BERT-emb)",
    os.path.join(OUT_DIR, "val_lr_bert.csv"),
    os.path.join(OUT_DIR, "test_lr_bert.csv")
)

if cal_rows:
    df_cal = pd.DataFrame(cal_rows)
    df_cal.to_csv(os.path.join(DATA_DIR, "calibration_baselines.csv"), index=False)
    print("Wrote", os.path.join(DATA_DIR, "calibration_baselines.csv"))
else:
    # Always emit a file (empty) so your paper scripts don't break
    pd.DataFrame([], columns=["model","AUROC","AUPRC","F1","ECE","Brier"]).to_csv(
        os.path.join(DATA_DIR, "calibration_baselines.csv"), index=False
    )
    print("[warn] No logits CSVs found; wrote empty data/calibration_baselines.csv")

print("\n✅ Done.")


Python: 3.10.14
Platform: Linux-4.18.0-553.64.1.el8_10.x86_64-x86_64-with-glibc2.34
transformers: 4.40.1
torch: 2.0.1+cu117
tenseal: 0.3.16
sklearn: 1.6.1
pandas: 2.2.2
numpy: 1.26.4
matplotlib: 3.10.1
Dataset size: 1055348
Embeddings shape: (50000, 768)

============ SEED 42 ============
A train: (7335, 768) | B_train: (30931, 768) | B_calib: (3866, 768) | B_test: (3867, 768) | Val: (10000, 768)
Epoch 01: train loss=0.6790, val loss=0.6681
Epoch 02: train loss=0.6549, val loss=0.6602
Epoch 03: train loss=0.6420, val loss=0.6495
Epoch 04: train loss=0.6343, val loss=0.6499
Epoch 05: train loss=0.6326, val loss=0.6463
Epoch 06: train loss=0.6246, val loss=0.6453
Epoch 07: train loss=0.6206, val loss=0.6424
Epoch 08: train loss=0.6144, val loss=0.6457
Epoch 09: train loss=0.6060, val loss=0.6513
Epoch 10: train loss=0.6058, val loss=0.6508
Early stopping triggered at epoch 10.
Epoch 01: train loss=0.6642, val loss=0.6430
Epoch 02: train loss=0.6380, val loss=0.6347
Epoch 03: train loss=0

# Here we can do averaging the plots

In [1]:
# ============================================
# BioClinicalBERT + Encrypted Last-Layer Calibration (CKKS) on MIMIC-III Notes
# One-cell end-to-end: training, calibration, multi-seed summary, per-seed summary, degree-sweep summary
# ============================================

import os, sys, re, time, platform, importlib, json, hashlib, warnings, math, glob
warnings.filterwarnings("ignore")
os.environ["TOKENIZERS_PARALLELISM"] = "false"  # avoid HF fork warnings

from typing import List, Tuple, Dict, Optional
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# -----------------------------
# 0) Reproducibility & settings
# -----------------------------
SEED = 42
np.random.seed(SEED)

def set_torch_seed(seed=SEED):
    try:
        import torch
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    except Exception:
        pass

def maybe_import(name):
    try: return importlib.import_module(name)
    except Exception: return None

# Soft deps
sklearn = maybe_import("sklearn")
tfm     = maybe_import("transformers")
torch_m = maybe_import("torch")
ts      = maybe_import("tenseal")

VERSIONS = {}
for pkg in ["transformers","torch","tenseal","sklearn","pandas","numpy","matplotlib"]:
    try:
        m = importlib.import_module(pkg)
        VERSIONS[pkg] = getattr(m, "__version__", "unknown")
    except Exception:
        VERSIONS[pkg] = "not installed"

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
for k,v in VERSIONS.items(): print(f"{k}: {v}")

# -----------------------------
# 1) Paths, toggles, parameters
# -----------------------------
DATA_PATH = "final_cohort_exact_email.csv.gz"  # expects HOURS_BEFORE_DEATH + NOTES_24HR or TEXT
OUT_DIR   = "q1_ckks_outputs"
FIG_DIR   = os.path.join(OUT_DIR, "figs")
os.makedirs(OUT_DIR, exist_ok=True); os.makedirs(FIG_DIR, exist_ok=True)

# Embedding strategy
USE_TFIDF_BASELINE   = False         # True → main model uses TF-IDF instead of BERT-emb
RUN_TFIDF_BASELINE_LR= True          # LR(TF-IDF) baseline
RUN_BERT_BASELINE_LR = True          # LR(BERT-emb) baseline

TFIDF_DIM    = 1024
BERT_MODEL   = "emilyalsentzer/Bio_ClinicalBERT"
MAX_DOCS     = 50_000               # dev cap; increase for final runs
MAX_TOKENS   = 256
BATCH_EMB    = 64

# Partition / training
VAL_CAP            = 10_000        # shared, balanced validation
PRIMARY_RATIO      = 0.85          # label-skew per site
EPOCHS             = 20
BATCH_TRAIN        = 512
LR                 = 1e-3
EARLY_STOP_PATIENCE= 3

# Head / Chebyshev
HIDDEN_DIM         = 64
CHEB_DEGREES       = [2,3,4,6]
DEFAULT_CHEB_COEFFS= {
    2: [0.5, 0.3, -0.1],
    3: [0.5, 0.25, -0.12, 0.05],
    4: [0.5, 0.25, -0.12, 0.05, -0.01],
    6: [0.5, 0.3, -0.2, 0.1, -0.05, 0.02, -0.005],
}
CHEB_COEFFS        = DEFAULT_CHEB_COEFFS[3]  # default
CALIB_STEPS     = 10         # was 20
CALIB_LR        = 0.002     # was 0.1
K_CALIB         = 60        # was 100            # # of misclassified B_calib samples to build the single calibrated head
CALIB_L2        = 1e-4      # new: mild L2 to stabilize W updates

# === Calibration policy switches ===
CALIB_BIAS_ONLY = False   # False -> update W and b  (NOT bias-only)
                          # True  -> update b only   (bias-only)
CALIB_POLICY    = "BiasOnly" if CALIB_BIAS_ONLY else "FullWb"


# Multi-seed runs for mean±std reporting
RUN_SEEDS          = [42,43,44,45,46]  # Q1 tables expect ≥5 seeds

# Misc
PLOT = True  # set False to skip plotting in headless environments

# CKKS params (log for reproducibility)
CKKS_PARAMS = dict(poly_modulus_degree=8192, coeff_mod_bit_sizes=[60,40,40,60], global_scale=2**40)

# -----------------------------
# 2) Load & clean MIMIC-III CSV
# -----------------------------
if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(f"{DATA_PATH} missing")

df_raw = pd.read_csv(DATA_PATH, compression="gzip", low_memory=False)

note_col = "NOTES_24HR" if "NOTES_24HR" in df_raw.columns else ("TEXT" if "TEXT" in df_raw.columns else None)
if note_col is None:
    raise ValueError("Missing notes column (NOTES_24HR or TEXT)")
if "HOURS_BEFORE_DEATH" not in df_raw.columns:
    raise ValueError("Missing HOURS_BEFORE_DEATH")

df = df_raw.copy()
df["survival_2day"] = np.where(df["HOURS_BEFORE_DEATH"].isna() | (df["HOURS_BEFORE_DEATH"] > 48), 1, 0)

def clean_text(txt: str) -> str:
    txt = str(txt).lower()
    txt = re.sub(r"\[.*?\]", " ", txt)
    txt = re.sub(r"[^a-z0-9\s]", " ", txt)
    return re.sub(r"\s+", " ", txt).strip()

df = df.dropna(subset=[note_col, "survival_2day"])
df["NOTES"] = df[note_col].astype(str).apply(clean_text)

labels_full = df["survival_2day"].astype(np.int64).values
texts_full  = df["NOTES"].tolist()
print("Dataset size:", len(texts_full))

# Subsample for dev speed
texts  = texts_full[:MAX_DOCS]
labels = labels_full[:MAX_DOCS]

# -----------------------------
# 3) Build/cached embeddings
# -----------------------------
def sha_short(s: str) -> str:
    return hashlib.sha256(s.encode()).hexdigest()[:10]

if USE_TFIDF_BASELINE:
    if sklearn is None: raise RuntimeError("scikit-learn not available for TF-IDF baseline")
    from sklearn.feature_extraction.text import TfidfVectorizer
    vec_tag = f"tfidf_{TFIDF_DIM}"
    emb_cache = os.path.join(OUT_DIR, f"{vec_tag}_{MAX_DOCS}docs.npy")
    if os.path.exists(emb_cache):
        X_all = np.load(emb_cache, mmap_mode="r")
    else:
        VEC = TfidfVectorizer(max_features=TFIDF_DIM, ngram_range=(1,2), min_df=5, max_df=0.95, sublinear_tf=True)
        X_all = VEC.fit_transform(texts).astype(np.float32).toarray()
        np.save(emb_cache, X_all)
    EMBED_DIM = X_all.shape[1]
    embed_manifest = dict(kind="tfidf", dim=EMBED_DIM, cache=emb_cache, sklearn=VERSIONS.get("sklearn"))
else:
    if tfm is None or torch_m is None:
        raise RuntimeError("transformers/torch not available for BERT embeddings")
    EMBED_DIM  = 768
    bert_tag = f"bioclinbert_{MAX_TOKENS}"
    emb_cache = os.path.join(OUT_DIR, f"{bert_tag}_{MAX_DOCS}docs.npy")

    def compute_llm_embeddings(texts: List[str]) -> np.ndarray:
        from transformers import AutoTokenizer, AutoModel
        device = "cuda" if torch_m.cuda.is_available() else "cpu"
        tok   = AutoTokenizer.from_pretrained(BERT_MODEL, use_fast=True)
        model = AutoModel.from_pretrained(BERT_MODEL).to(device).eval()
        feats = []
        with torch_m.no_grad():
            for i in range(0, len(texts), BATCH_EMB):
                batch = texts[i:i+BATCH_EMB]
                enc = tok(batch, truncation=True, padding=True, max_length=MAX_TOKENS, return_tensors="pt").to(device)
                out = model(**enc).last_hidden_state[:, 0, :].cpu().numpy()
                feats.append(out)
        return np.vstack(feats).astype(np.float32)

    if os.path.exists(emb_cache):
        X_all = np.load(emb_cache, mmap_mode="r")
    else:
        X_all = compute_llm_embeddings(texts)
        np.save(emb_cache, X_all)
    embed_manifest = dict(kind="bioclinicalbert", dim=EMBED_DIM, cache=emb_cache, model=BERT_MODEL, transformers=VERSIONS.get("transformers"))

print("Embeddings shape:", X_all.shape)

# -----------------------------
# 4) Utilities (metrics, plots)
# -----------------------------
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support, roc_auc_score, average_precision_score,
    confusion_matrix, roc_curve, precision_recall_curve, brier_score_loss
)

def expected_calibration_error(y_true, prob, n_bins=15):
    y_true = np.asarray(y_true).astype(int)
    prob = np.asarray(prob).astype(float)
    bins = np.linspace(0.0, 1.0, n_bins+1)
    inds = np.digitize(prob, bins) - 1
    ece = 0.0
    for b in range(n_bins):
        mask = inds == b
        if not np.any(mask): continue
        conf = prob[mask].mean()
        acc  = (y_true[mask] == (prob[mask] >= 0.5)).mean()
        ece += (np.sum(mask)/len(prob)) * abs(acc - conf)
    return float(ece)

def reliability_plot(tag, y_true, prob, fig_dir=FIG_DIR, n_bins=15):
    y_true = np.asarray(y_true).astype(int)
    prob = np.asarray(prob).astype(float)
    bins = np.linspace(0.0, 1.0, n_bins+1)
    inds = np.digitize(prob, bins) - 1
    xs, ys = [], []
    for b in range(n_bins):
        mask = inds == b
        if not np.any(mask): continue
        xs.append(prob[mask].mean())
        ys.append((y_true[mask] == (prob[mask] >= 0.5)).mean())
    if PLOT:
        plt.figure()
        plt.plot([0,1],[0,1],'--')
        plt.plot(xs, ys, marker='o')
        plt.xlabel("Confidence")
        plt.ylabel("Empirical Accuracy")
        plt.title(f"Reliability: {tag} (ECE={expected_calibration_error(y_true, prob, n_bins):.3f})")
        plt.tight_layout()
        fn = os.path.join(fig_dir, f"reliability_{tag}.png")
        plt.savefig(fn); plt.show()

def predict_probs(model, X, batch_size=1024, device="cpu"):
    model.eval()
    ds = TensorDataset(torch.tensor(X, dtype=torch.float32))
    dl = DataLoader(ds, batch_size=batch_size, shuffle=False)
    probs = []
    with torch.no_grad():
        for (xb,) in dl:
            xb = xb.to(device)
            logits, _ = model(xb)
            p = torch.softmax(logits, dim=1)[:, 1].cpu().numpy()
            probs.append(p)
    return np.concatenate(probs, axis=0)

def metrics_at(y, prob, t=0.5):
    y = np.asarray(y).astype(int); prob = np.asarray(prob).astype(float)
    yhat = (prob >= t).astype(int)
    prec, rec, f1, _ = precision_recall_fscore_support(y, yhat, average="binary", zero_division=0)
    out = dict(
        Acc=accuracy_score(y, yhat),
        Prec=prec, Rec=rec, F1=f1,
        AUROC=roc_auc_score(y, prob),
        AUPRC=average_precision_score(y, prob),
        Brier=brier_score_loss(y, prob)
    )
    cm = confusion_matrix(y, yhat).tolist()
    out["ConfMat"] = cm
    return out

def best_threshold(y, prob, metric="F1"):
    ts = np.linspace(0.05, 0.95, 19)
    best_t, best_v = 0.5, -1.0
    for t in ts:
        v = metrics_at(y, prob, t)[metric]
        if not np.isnan(v) and v > best_v:
            best_v, best_t = v, t
    return best_t, best_v

def dump_curves(tag, y_true, prob, out_dir=OUT_DIR, fig_dir=FIG_DIR):
    fpr, tpr, _ = roc_curve(y_true, prob)
    prec, rec, _ = precision_recall_curve(y_true, prob)
    pd.DataFrame({"fpr": fpr, "tpr": tpr}).to_csv(os.path.join(out_dir, f"roc_{tag}.csv"), index=False)
    pd.DataFrame({"precision": prec, "recall": rec}).to_csv(os.path.join(out_dir, f"pr_{tag}.csv"), index=False)
    if PLOT:
        plt.figure(); plt.plot(fpr, tpr)
        plt.xlabel("FPR"); plt.ylabel("TPR"); plt.title(f"ROC: {tag}")
        plt.tight_layout(); plt.savefig(os.path.join(fig_dir, f"roc_{tag}.png")); plt.show()
        plt.figure(); plt.plot(rec, prec)
        plt.xlabel("Recall"); plt.ylabel("Precision"); plt.title(f"PR: {tag}")
        plt.tight_layout(); plt.savefig(os.path.join(fig_dir, f"pr_{tag}.png")); plt.show()

# =========================================================
# 4B) Multiseed plotting helpers for paper-ready figures
# =========================================================
from sklearn.metrics import roc_curve, precision_recall_curve, auc

def _bin_reliability(y_true, prob, n_bins=15):
    y_true = np.asarray(y_true).astype(int)
    prob = np.asarray(prob).astype(float)

    bins = np.linspace(0.0, 1.0, n_bins + 1)
    inds = np.digitize(prob, bins) - 1

    xs, ys = [], []
    for b in range(n_bins):
        mask = inds == b
        if not np.any(mask):
            continue
        xs.append(prob[mask].mean())
        ys.append((y_true[mask] == (prob[mask] >= 0.5)).mean())
    return np.array(xs), np.array(ys)

def plot_multiseed_reliability(all_runs, fig_dir=FIG_DIR, n_bins=15):
    """
    Creates one merged figure with 3 panels:
      (a) Pre
      (b) Post
      (c) Post+TS
    using mean ± std across seeds.
    """
    os.makedirs(fig_dir, exist_ok=True)

    stages = [
        ("Pre", "btest_probs_pre"),
        ("Post", "btest_probs_post"),
        ("Post+TS", "btest_probs_postTS"),
    ]

    fig, axes = plt.subplots(1, 3, figsize=(12, 4))

    for ax, (title, key) in zip(axes, stages):
        grid = np.linspace(0.0, 1.0, 101)
        all_interp = []

        for r in all_runs:
            y = np.asarray(r["btest_y"]).astype(int)
            p = np.asarray(r[key]).astype(float)

            xs, ys = _bin_reliability(y, p, n_bins=n_bins)
            if len(xs) < 2:
                continue

            order = np.argsort(xs)
            xs = xs[order]
            ys = ys[order]

            interp_y = np.interp(grid, xs, ys, left=np.nan, right=np.nan)
            all_interp.append(interp_y)

        if len(all_interp) == 0:
            ax.plot([0, 1], [0, 1], "--")
            ax.set_title(title)
            ax.set_xlabel("Confidence")
            ax.set_ylabel("Empirical Accuracy")
            continue

        arr = np.array(all_interp, dtype=float)
        mean_y = np.nanmean(arr, axis=0)
        std_y  = np.nanstd(arr, axis=0)

        ax.plot([0, 1], [0, 1], "--", linewidth=1)
        ax.plot(grid, mean_y, linewidth=2)
        ax.fill_between(grid, mean_y - std_y, mean_y + std_y, alpha=0.2)

        # global ECE summary for title
        eces = []
        for r in all_runs:
            y = np.asarray(r["btest_y"]).astype(int)
            p = np.asarray(r[key]).astype(float)
            eces.append(expected_calibration_error(y, p, n_bins=n_bins))
        ece_mean = np.mean(eces)
        ece_std  = np.std(eces)

        ax.set_title(f"{title} (ECE={ece_mean:.3f}±{ece_std:.3f})")
        ax.set_xlabel("Confidence")
        ax.set_ylabel("Empirical Accuracy")

    plt.tight_layout()
    out_path = os.path.join(fig_dir, "reliability_Btest_multiseed_triptych.png")
    plt.savefig(out_path, dpi=300, bbox_inches="tight")
    plt.show()

def mean_roc_curve(items, fig_path, title="ROC (multiseed)"):
    """
    items: list of dicts with keys:
        { "y": [...], "prob": [...] }
    """
    os.makedirs(os.path.dirname(fig_path), exist_ok=True)

    grid = np.linspace(0.0, 1.0, 501)
    curves = []
    aucs = []

    for it in items:
        y = np.asarray(it["y"]).astype(int)
        p = np.asarray(it["prob"]).astype(float)

        fpr, tpr, _ = roc_curve(y, p)
        interp_tpr = np.interp(grid, fpr, tpr)
        interp_tpr[0] = 0.0
        interp_tpr[-1] = 1.0
        curves.append(interp_tpr)
        aucs.append(auc(fpr, tpr))

    curves = np.array(curves)
    mean_tpr = curves.mean(axis=0)
    std_tpr  = curves.std(axis=0)

    plt.figure(figsize=(5, 4))
    plt.plot([0, 1], [0, 1], "--", linewidth=1)
    plt.plot(grid, mean_tpr, linewidth=2, label=f"Mean ROC (AUROC={np.mean(aucs):.3f}±{np.std(aucs):.3f})")
    plt.fill_between(
        grid,
        np.maximum(mean_tpr - std_tpr, 0.0),
        np.minimum(mean_tpr + std_tpr, 1.0),
        alpha=0.2
    )
    plt.xlabel("FPR")
    plt.ylabel("TPR")
    plt.title(title)
    plt.legend(loc="lower right")
    plt.tight_layout()
    plt.savefig(fig_path, dpi=300, bbox_inches="tight")
    plt.show()

def mean_pr_curve(items, fig_path, title="PR (multiseed)"):
    """
    items: list of dicts with keys:
        { "y": [...], "prob": [...] }
    """
    os.makedirs(os.path.dirname(fig_path), exist_ok=True)

    grid = np.linspace(0.0, 1.0, 501)
    curves = []
    pr_aucs = []

    for it in items:
        y = np.asarray(it["y"]).astype(int)
        p = np.asarray(it["prob"]).astype(float)

        precision, recall, _ = precision_recall_curve(y, p)

        order = np.argsort(recall)
        recall = recall[order]
        precision = precision[order]

        interp_prec = np.interp(grid, recall, precision)
        curves.append(interp_prec)
        pr_aucs.append(auc(recall, precision))

    curves = np.array(curves)
    mean_prec = curves.mean(axis=0)
    std_prec  = curves.std(axis=0)

    plt.figure(figsize=(5, 4))
    plt.plot(grid, mean_prec, linewidth=2, label=f"Mean PR (AUPRC={np.mean(pr_aucs):.3f}±{np.std(pr_aucs):.3f})")
    plt.fill_between(
        grid,
        np.maximum(mean_prec - std_prec, 0.0),
        np.minimum(mean_prec + std_prec, 1.0),
        alpha=0.2
    )
    plt.xlabel("Recall")
    plt.ylabel("Precision")
    plt.title(title)
    plt.legend(loc="lower left")
    plt.tight_layout()
    plt.savefig(fig_path, dpi=300, bbox_inches="tight")
    plt.show()

def plot_multiseed_degree_sweep(out_dir=OUT_DIR, fig_dir=FIG_DIR):
    """
    Reads all per-seed degree sweep CSVs and creates one mean±std figure.
    """
    os.makedirs(fig_dir, exist_ok=True)

    paths = sorted(glob.glob(os.path.join(out_dir, "chebyshev_degree_sweep_seed*.csv")))
    if len(paths) == 0:
        print("[warn] No degree-sweep CSVs found.")
        return

    dfs = []
    for p in paths:
        try:
            seed = int(os.path.basename(p).split("seed")[-1].split(".csv")[0])
        except Exception:
            seed = -1
        d = pd.read_csv(p)
        d["seed"] = seed
        dfs.append(d)

    df_all = pd.concat(dfs, ignore_index=True)

    agg = (df_all
           .groupby("degree")
           .agg(AUROC_mean=("AUROC", "mean"),
                AUROC_std=("AUROC", "std"))
           .reset_index())

    plt.figure(figsize=(5, 4))
    plt.plot(agg["degree"], agg["AUROC_mean"], marker="o", linewidth=2)
    plt.fill_between(
        agg["degree"],
        agg["AUROC_mean"] - agg["AUROC_std"].fillna(0.0),
        agg["AUROC_mean"] + agg["AUROC_std"].fillna(0.0),
        alpha=0.2
    )
    plt.xlabel("Chebyshev degree")
    plt.ylabel("AUROC")
    plt.title("Chebyshev degree sweep (multiseed)")
    plt.tight_layout()
    plt.savefig(os.path.join(fig_dir, "cheb_degree_auroc_multiseed.png"), dpi=300, bbox_inches="tight")
    plt.show()

# -----------------------------
# 5) Model head & training
# -----------------------------
device = "cuda" if (torch_m is not None and torch_m.cuda.is_available()) else "cpu"

class SimpleHead(nn.Module):
    def __init__(self, input_dim, hidden=HIDDEN_DIM, num_classes=2, p_drop=0.10):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, hidden)
        self.drop= nn.Dropout(p_drop)
        self.fc2 = nn.Linear(hidden, num_classes)  # logits
    def forward(self, x):
        x = torch.relu(self.fc1(x))
        feat = self.drop(x)
        return self.fc2(feat), feat  # logits, penultimate

def class_weights_from_labels(y: np.ndarray) -> torch.Tensor:
    classes, counts = np.unique(y, return_counts=True)
    total = counts.sum()
    weights = total / (len(classes) * counts.astype(np.float32))
    w = torch.ones(2, dtype=torch.float32)
    for c, wt in zip(classes, weights): w[int(c)] = float(wt)
    return w

def train_head(X_train, y_train, X_val, y_val, input_dim, epochs=EPOCHS, batch_size=BATCH_TRAIN, lr=LR, patience=EARLY_STOP_PATIENCE):
    model = SimpleHead(input_dim=input_dim).to(device)
    weight = class_weights_from_labels(y_train).to(device)
    lossfn = nn.CrossEntropyLoss(weight=weight)
    opt = optim.Adam(model.parameters(), lr=lr)

    ds_tr = TensorDataset(torch.tensor(X_train, dtype=torch.float32), torch.tensor(y_train, dtype=torch.long))
    ds_va = TensorDataset(torch.tensor(X_val,   dtype=torch.float32), torch.tensor(y_val,   dtype=torch.long))
    dl_tr = DataLoader(ds_tr, batch_size=batch_size, shuffle=True,  drop_last=False)
    dl_va = DataLoader(ds_va, batch_size=batch_size, shuffle=False, drop_last=False)

    best_val = float("inf"); best_state = None; wait = 0

    for epoch in range(epochs):
        model.train(); running = 0.0
        for xb, yb in dl_tr:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            logits, _ = model(xb)
            loss = lossfn(logits, yb)
            loss.backward(); opt.step()
            running += loss.item() * xb.size(0)
        tr_loss = running / len(ds_tr)

        model.eval(); v_running = 0.0
        with torch.no_grad():
            for xb, yb in dl_va:
                xb, yb = xb.to(device), yb.to(device)
                logits, _ = model(xb)
                v_running += lossfn(logits, yb).item() * xb.size(0)
        va_loss = v_running / len(ds_va)
        print(f"Epoch {epoch+1:02d}: train loss={tr_loss:.4f}, val loss={va_loss:.4f}")

        if va_loss + 1e-6 < best_val:
            best_val = va_loss
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            wait = 0
        else:
            wait += 1
            if wait >= patience:
                print(f"Early stopping triggered at epoch {epoch+1}.")
                break

    if best_state is not None:
        model.load_state_dict(best_state)
    return model

# -----------------------------
# 6) Chebyshev softmax (HE-friendly; clamp to fixed L from validation)
# -----------------------------
def cheb_softmax_clamped(logits_row: np.ndarray, coeffs: List[float], L: float) -> np.ndarray:
    # 1) clamp logits to [-L, L]
    z = np.clip(logits_row, -L, L)
    # 2) map to [-1, 1]
    x = z / (L + 1e-12)
    # 3) Chebyshev series
    T0 = np.ones_like(x)
    T1 = x.copy()
    T  = [T0, T1]
    for _ in range(2, len(coeffs)):
        Tn = 2 * x * T[-1] - T[-2]
        T.append(Tn)
    poly = sum(c * Tn for c, Tn in zip(coeffs, T))
    poly = np.clip(poly, 1e-6, np.inf)
    return poly / np.sum(poly)

# Backward-compat wrapper:
def cheb_softmax(logits_row: np.ndarray, coeffs: List[float], L: Optional[float] = None) -> np.ndarray:
    assert L is not None, "Provide L from validation for clamped Chebyshev softmax."
    return cheb_softmax_clamped(logits_row, coeffs, L)

# -----------------------------
# 7) One SEED run (returns dict of outputs)
# -----------------------------
from sklearn.model_selection import train_test_split

def run_one_seed(seed:int=42) -> Dict:
    set_torch_seed(seed)
    rng = np.random.default_rng(seed)

    # shared 50/50 validation (balanced)
    pos_idx, neg_idx = np.where(labels==1)[0], np.where(labels==0)[0]
    n_val = min(VAL_CAP//2, len(pos_idx), len(neg_idx))
    val_idx = np.concatenate([rng.choice(pos_idx, n_val, replace=False),
                              rng.choice(neg_idx, n_val, replace=False)])
    train_mask = np.ones(len(labels), bool); train_mask[val_idx] = False

    X_val, y_val = X_all[val_idx], labels[val_idx]
    X_pool, y_pool = X_all[train_mask], labels[train_mask]

    def skew(X, y, primary_labels, primary_ratio=PRIMARY_RATIO, seed=seed):
        r = np.random.default_rng(seed)
        mask = np.isin(y, primary_labels)
        Xp, yp = X[mask], y[mask]
        Xs, ys = X[~mask], y[~mask]
        k = int((1-primary_ratio) * len(Xp)); k = min(max(k,0), len(Xs))
        idx = r.choice(len(Xs), k, replace=False) if k>0 else np.array([], int)
        Xf = np.concatenate([Xp, Xs[idx]], 0)
        yf = np.concatenate([yp, ys[idx]], 0)
        perm = r.permutation(len(Xf))
        return Xf[perm], yf[perm]

    X_A,     y_A     = skew(X_pool, y_pool, [0], seed=seed)     # A enriched for class 0
    X_B_all, y_B_all = skew(X_pool, y_pool, [1], seed=seed+1)   # B enriched for class 1

    # First: Train / (Calib+Test)
    X_B_train, X_B_rest, y_B_train, y_B_rest = train_test_split(
        X_B_all, y_B_all, test_size=0.2, stratify=y_B_all, random_state=seed+2
    )
    # Second: split the 20% into half Calib, half Test (10%/10% overall)
    X_B_calib, X_B_test, y_B_calib, y_B_test = train_test_split(
        X_B_rest, y_B_rest, test_size=0.5, stratify=y_B_rest, random_state=seed+3
    )
    print(f"A train: {X_A.shape} | B_train: {X_B_train.shape} | B_calib: {X_B_calib.shape} | B_test: {X_B_test.shape} | Val: {X_val.shape}")

    # Train heads
    mA = train_head(X_A, y_A, X_val, y_val, input_dim=X_A.shape[1])
    mB = train_head(X_B_train, y_B_train, X_val, y_val, input_dim=X_B_train.shape[1])

    # Validation penultimate via mB (to match test path)
    with torch.no_grad():
        ds_val = TensorDataset(torch.tensor(X_val, dtype=torch.float32))
        dl_val = DataLoader(ds_val, batch_size=1024, shuffle=False)
        feats_val = []
        mB.eval()  # <<< ADD THIS
        for (xb,) in dl_val:
            xb = xb.to(device)
            _, f = mB(xb)
            feats_val.append(f.cpu().numpy())

    F_val = np.vstack(feats_val)  # (N_val, 64)

    # Fix Chebyshev clamp L on validation logits (robust to outliers)
    W_base = mA.fc2.weight.detach().cpu().numpy().T   # (64,2)
    b_base = mA.fc2.bias.detach().cpu().numpy()
    logits_val = F_val @ W_base + b_base
    L_CHEB = float(np.percentile(np.abs(logits_val), 99.0, axis=None))
    L_CHEB = max(L_CHEB, 1.0)


    # One frozen threshold t* from validation (using A-head over F_val)
    probs_val_pre = np.array([cheb_softmax(row, CHEB_COEFFS, L=L_CHEB)[1] for row in logits_val])
    t_star, _ = best_threshold(y_val, probs_val_pre, metric="F1")

    # === NEW: dump A-head logits for validation ===
    import pandas as _pd, numpy as _np
    _ahead_val_logits = (logits_val[:,1] - logits_val[:,0]).ravel()  # logit = z1 - z0
    _pd.DataFrame({"logit": _np.asarray(_ahead_val_logits), "y": y_val.astype(int)}).to_csv(
        os.path.join(OUT_DIR, "ahead_val_logits.csv"), index=False
    )


    # Baselines (optional LR on embeddings)
    baseline_rows=[]
    if sklearn is not None:
        from sklearn.linear_model import LogisticRegression

        def eval_lr(X_tr, y_tr, X_te, y_te, tag):
            lr_clf = LogisticRegression(max_iter=1000, solver="lbfgs", class_weight="balanced", n_jobs=-1)
            lr_clf.fit(X_tr, y_tr)
            prob = lr_clf.predict_proba(X_te)[:,1]
            t, _ = best_threshold(y_te, prob, metric="F1")
            yhat = (prob >= t).astype(int)
            row = dict(
                model=tag, AUROC=float(roc_auc_score(y_te, prob)),
                AUPRC=float(average_precision_score(y_te, prob)),
                F1=float(precision_recall_fscore_support(y_te, yhat, average="binary", zero_division=0)[2]),
                Acc=float(accuracy_score(y_te, yhat)), t_star=float(t)
            )
            return row

        if RUN_BERT_BASELINE_LR and embed_manifest["kind"] == "bioclinicalbert":
            baseline_rows.append(eval_lr(X_pool, y_pool, X_val, y_val, f"LR(BERT-emb)_seed{seed}"))
        if RUN_TFIDF_BASELINE_LR:
            try:
                from sklearn.feature_extraction.text import TfidfVectorizer
                VEC_BL = TfidfVectorizer(max_features=TFIDF_DIM, ngram_range=(1,2), min_df=5, max_df=0.95, sublinear_tf=True)
                X_all_tfidf = VEC_BL.fit_transform(texts).astype(np.float32).toarray()
                X_val_tf = X_all_tfidf[val_idx]
                X_pool_tf= X_all_tfidf[train_mask]
                y_pool_tf= y_pool
                baseline_rows.append(eval_lr(X_pool_tf, y_pool_tf, X_val_tf, y_val, f"LR(TF-IDF)_seed{seed}"))
            except Exception as e:
                print("[warn] TF-IDF LR baseline skipped:", e)

    # Helper to get penultimate features
    def penultimate_from_array(X_arr, idx):
        with torch.no_grad():
            mB.eval()  # <<< ADD THIS
            xb = torch.tensor(X_arr[idx].reshape(1, -1), dtype=torch.float32).to(device)
            _, feat = mB(xb)
        return feat.cpu().numpy()[0]


    # Select misclassified indices under A-head (Chebyshev + clamp L_CHEB)
    def pick_misclassified_indices(max_k=K_CALIB, coeffs=CHEB_COEFFS):
        """
        Build a class-balanced set of the hardest misclassified examples (smallest margin).
        """
        cand = {0: [], 1: []}  # store (margin, idx) pairs per class
        for i in range(len(X_B_calib)):
            f = penultimate_from_array(X_B_calib, i)
            p = cheb_softmax(f @ W_base + b_base, coeffs, L=L_CHEB)
            y_i = int(y_B_calib[i])
            y_hat = int(np.argmax(p))
            if y_hat != y_i:
                # hardness = distance from decision: smaller |p1-0.5| => harder
                margin = abs(p[1] - 0.5)
                cand[y_i].append((margin, i))
    
        # sort by hardness (ascending margin → harder)
        for c in (0, 1):
            cand[c].sort(key=lambda t: t[0])
    
        # balanced take
        per_class = max_k // 2
        sel = [idx for _, idx in cand[0][:per_class]] + [idx for _, idx in cand[1][:per_class]]
    
        # if one side runs short, fill from the other (still hardest first)
        if len(sel) < max_k:
            remaining = max_k - len(sel)
            pool = cand[0][per_class:] + cand[1][per_class:]
            pool.sort(key=lambda t: t[0])
            sel += [idx for _, idx in pool[:remaining]]
    
        return sel[:max_k]


    mis_idx = pick_misclassified_indices(K_CALIB, CHEB_COEFFS)
    print(f"Selected {len(mis_idx)} misclassified B_calib samples for calibration batch (seed={seed}).")

    # Optional HE timings + size
    he_timings = {}; he_payload_bytes=None
    if ts is not None and len(mis_idx)>0:
        try:
            f0 = penultimate_from_array(X_B_calib, mis_idx[0])
            t0 = time.time()
            context = ts.context(ts.SCHEME_TYPE.CKKS, poly_modulus_degree=CKKS_PARAMS["poly_modulus_degree"],
                                 coeff_mod_bit_sizes=CKKS_PARAMS["coeff_mod_bit_sizes"])
            context.global_scale = CKKS_PARAMS["global_scale"]
            context.generate_galois_keys()
            he_timings["context_ms"] = (time.time()-t0)*1000

            t1=time.time(); enc_feat = ts.ckks_vector(context, f0.tolist())
            he_timings["encrypt_feat_ms"]=(time.time()-t1)*1000
            try: he_payload_bytes = len(enc_feat.serialize())
            except Exception: he_payload_bytes=None

            t2=time.time(); logits_he=[]
            for j in range(W_base.shape[1]):
                lj = enc_feat.dot(W_base[:, j].tolist()).decrypt()[0] + float(b_base[j])
                logits_he.append(lj)
            he_timings["enc_dot_total_ms"]=(time.time()-t2)*1000
            print("HE warmup logits (decrypted):", np.round(logits_he,4))
        except Exception as e:
            print("[warn] HE timing step skipped:", e)

    # ------- Build a SINGLE calibrated head from the misclassified batch -------
    W_cal = W_base.copy(); b_cal = b_base.copy()
    W_base_copy = W_base.copy()
    b_base_copy = b_base.copy()

    def calibrate_once(f, ytrue, coeffs=CHEB_COEFFS, steps=CALIB_STEPS, lr=CALIB_LR, 
                   W=None, b=None, bias_only=CALIB_BIAS_ONLY):
        """
        One-sample calibration loop with mild L2 regularization to base params.
        """
        if W is None: W = W_cal
        if b is None: b = b_cal
    
        target = np.array([1.0 if k==ytrue else 0.0 for k in range(2)], dtype=float)
    
        for _ in range(steps):
            p = cheb_softmax(f @ W + b, coeffs, L=L_CHEB)  # (2,)
            err = target - p
    
            if bias_only:
                for j in range(W.shape[1]):
                    b[j] += lr * (err[j] - CALIB_L2 * (b[j] - b_base_copy[j]))
            else:
                for j in range(W.shape[1]):
                    W[:, j] += lr * (err[j] * f - CALIB_L2 * (W[:, j] - W_base_copy[:, j]))
                    b[j]    += lr * (err[j] - CALIB_L2 * (b[j] - b_base_copy[j]))
        return W, b



    t_start = time.time()
    for idx_m in mis_idx:
        f = penultimate_from_array(X_B_calib, idx_m)
        ytrue = int(y_B_calib[idx_m])
        W_cal, b_cal = calibrate_once(
            f, ytrue, CHEB_COEFFS, CALIB_STEPS, CALIB_LR, 
            W_cal, b_cal, bias_only=CALIB_BIAS_ONLY
        )
    calib_batch_ms = (time.time()-t_start)*1000

    # ------- Evaluate PRE/POST calibration on FULL B_test -------
    feats = []
    with torch.no_grad():
        ds = TensorDataset(torch.tensor(X_B_test, dtype=torch.float32))
        dl = DataLoader(ds, batch_size=1024, shuffle=False)
        mB.eval()  # <<< ADD THIS
        for (xb,) in dl:
            xb = xb.to(device)
            _, f = mB(xb)
            feats.append(f.cpu().numpy())
    F_B = np.vstack(feats)  # (N,64)


    # === NEW: dump A-head logits for B_test ===
    logits_btest_ahead = F_B @ W_base + b_base
    _ahead_btest_logits = (logits_btest_ahead[:,1] - logits_btest_ahead[:,0]).ravel()
    _pd.DataFrame({"logit": _np.asarray(_ahead_btest_logits), "y": y_B_test.astype(int)}).to_csv(
        os.path.join(OUT_DIR, "ahead_btest_logits.csv"), index=False
    )


    def probs_from_params(F, W, b, coeffs=CHEB_COEFFS):
        logits = F @ W + b
        out = np.empty((len(F),), dtype=float)
        for i in range(len(F)):
            out[i] = cheb_softmax(logits[i], coeffs, L=L_CHEB)[1]  # prob of class 1
        return out

    probs_pre  = probs_from_params(F_B, W_base, b_base, CHEB_COEFFS)
    probs_post = probs_from_params(F_B, W_cal,  b_cal,  CHEB_COEFFS)
    # --- Temperature scaling on validation, applied to post-calibration logits ---
    # Build validation logits from calibrated head
    logits_val_post = F_val @ W_cal + b_cal
    # Convert 2-class logits to a single logit = z1 - z0
    logit_val_1d = (logits_val_post[:, 1] - logits_val_post[:, 0]).ravel()
    
    def _sigmoid(z): 
        return 1.0 / (1.0 + np.exp(-z))
    
    def _fit_temperature(lv, yv):
        t_candidates = np.linspace(0.5, 5.0, 91)
        best_t, best_nll = 1.0, 1e9
        yv = yv.astype(float)
        eps = 1e-9
        for t in t_candidates:
            p = _sigmoid(lv / t)
            nll = -(yv*np.log(p+eps) + (1-yv)*np.log(1-p+eps)).mean()
            if nll < best_nll:
                best_t, best_nll = t, nll
        return best_t
    
    T_post = _fit_temperature(logit_val_1d, y_val)
    
    # Apply T to B_test logits from calibrated head
    logits_btest_post = F_B @ W_cal + b_cal
    logit_btest_1d = (logits_btest_post[:, 1] - logits_btest_post[:, 0]).ravel()
    probs_post_ts = _sigmoid(logit_btest_1d / T_post)


    # Single frozen threshold picked on validation (t_star)
    mets_pre   = metrics_at(y_B_test, probs_pre,  t_star)
    mets_post  = metrics_at(y_B_test, probs_post, t_star)
    mets_post_ts = metrics_at(y_B_test, probs_post_ts, t_star)
    
    ece_pre     = expected_calibration_error(y_B_test, probs_pre)
    ece_post    = expected_calibration_error(y_B_test, probs_post)
    ece_post_ts = expected_calibration_error(y_B_test, probs_post_ts)

    # if PLOT:
    #     reliability_plot(f"Btest_pre_seed{seed}",  y_B_test, probs_pre)
    #     reliability_plot(f"Btest_post_seed{seed}", y_B_test, probs_post)
    #     reliability_plot(f"Btest_postTS_seed{seed}", y_B_test, probs_post_ts)  # <<< NEW

    # Persist per-seed calibration JSON
    with open(os.path.join(OUT_DIR, f"Btest_calibration_seed{seed}.json"), "w") as fjs:
        json.dump({
        "policy": CALIB_POLICY,
        "pre":      dict(threshold=float(t_star), **mets_pre,      ECE=float(ece_pre)),
        "post":     dict(threshold=float(t_star), **mets_post,     ECE=float(ece_post)),
        "post_TS":  dict(threshold=float(t_star), **mets_post_ts,  ECE=float(ece_post_ts), T=float(T_post)),
        "calibration_batch_ms": float(calib_batch_ms),
        "num_misclassified_used": int(len(mis_idx)),
        "L_CHEB": float(L_CHEB)
    }, fjs, indent=2)



    # Degree sweep (on B_test via A-head) — uses clamped Chebyshev (L_CHEB)
    def degree_sweep(degrees=CHEB_DEGREES):
        results = []
        logits = F_B @ W_base + b_base
        for d in degrees:
            coeffs = DEFAULT_CHEB_COEFFS[d]
            t0 = time.time()
            probs = np.array([cheb_softmax(row, coeffs, L=L_CHEB)[1] for row in logits])
            sweep_ms = (time.time()-t0)*1000
            t_deg, _ = best_threshold(y_B_test, probs, "F1")  # per-degree diagnostic threshold
            mets = metrics_at(y_B_test, probs, t_deg)
            results.append(dict(
                degree=d, AUROC=float(mets["AUROC"]), AUPRC=float(mets["AUPRC"]),
                F1=float(mets["F1"]), Acc=float(mets["Acc"]),
                t_star=float(t_deg), time_ms=float(sweep_ms),
                ECE=float(expected_calibration_error(y_B_test, probs))
            ))
        return pd.DataFrame(results)

    deg_df = degree_sweep(CHEB_DEGREES)
    deg_df.to_csv(os.path.join(OUT_DIR, f"chebyshev_degree_sweep_seed{seed}.csv"), index=False)
    
    # Keep per-seed CSVs, but do not create single-seed paper figure
    # if PLOT:
    #     plt.figure(); plt.plot(deg_df["degree"], deg_df["AUROC"], marker="o")
    #     plt.xlabel("Chebyshev degree"); plt.ylabel("AUROC")
    #     plt.title(f"Chebyshev Degree Sweep (seed={seed})")
    #     plt.tight_layout(); plt.savefig(os.path.join(FIG_DIR, f"cheb_degree_auroc_seed{seed}.png")); plt.show()

    # Validation summary payload (kept for completeness)
    valA = predict_probs(mA, X_val, device=device);  tA, _ = best_threshold(y_val, valA, metric="F1")
    valB = predict_probs(mB, X_val, device=device);  tB, _ = best_threshold(y_val, valB, metric="F1")
    mA_05 = metrics_at(y_val, valA, 0.5); mA_t = metrics_at(y_val, valA, tA)
    mB_05 = metrics_at(y_val, valB, 0.5); mB_t = metrics_at(y_val, valB, tB)

    # Keep scalar validation metrics, but do not create paper-facing single-seed figures
    # dump_curves(f"A_val_seed{seed}", y_val, valA)
    # dump_curves(f"B_val_seed{seed}", y_val, valB)
    # reliability_plot(f"A_val_seed{seed}", y_val, valA)
    # reliability_plot(f"B_val_seed{seed}", y_val, valB)


    val_payload = dict(
        A_at_0p5=mA_05, A_at_tstar=dict(t=float(tA), **mA_t),
        B_at_0p5=mB_05, B_at_tstar=dict(t=float(tB), **mB_t),
    )

    return {
        "seed": seed,
        "splits": dict(A=X_A.shape, Btr=X_B_train.shape, Bte=X_B_test.shape, Val=X_val.shape),
        "val": val_payload,
        "baselines": baseline_rows,
        "he": dict(params=CKKS_PARAMS, timings_ms=he_timings, payload_bytes=he_payload_bytes),
    
        "btest_calibration": {
            "pre": dict(t=float(t_star), **mets_pre, ECE=float(ece_pre)),
            "post": dict(t=float(t_star), **mets_post, ECE=float(ece_post)),
            "post_TS": dict(t=float(t_star), **mets_post_ts, ECE=float(ece_post_ts), T=float(T_post)),
            "batch_ms": float(calib_batch_ms),
            "k_used": int(len(mis_idx)),
            "L_CHEB": float(L_CHEB)
        },
    
        "degree_sweep_path": os.path.join(OUT_DIR, f"chebyshev_degree_sweep_seed{seed}.csv"),
    
        # needed for multiseed paper figures
        "btest_y": y_B_test.tolist(),
        "btest_probs_pre": probs_pre.tolist(),
        "btest_probs_post": probs_post.tolist(),
        "btest_probs_postTS": probs_post_ts.tolist(),
        "aval_y": y_val.tolist(),
        "aval_probs_pre": valA.tolist()
    }

# -----------------------------
# 8) Run multi-seed + aggregate
# -----------------------------
all_runs = []
for s in RUN_SEEDS:
    print("\n" + "="*12 + f" SEED {s} " + "="*12)
    out = run_one_seed(s)
    all_runs.append(out)

# ================================================
# Paper-ready multiseed figures
# ================================================
roc_items = []
pr_items = []

for r in all_runs:
    roc_items.append({
        "y": r["aval_y"],
        "prob": r["aval_probs_pre"]
    })
    pr_items.append({
        "y": r["aval_y"],
        "prob": r["aval_probs_pre"]
    })

# 1) Multiseed reliability triptych for B_test
plot_multiseed_reliability(all_runs, fig_dir=FIG_DIR, n_bins=15)

# 2) Multiseed ROC on shared validation
mean_roc_curve(
    roc_items,
    fig_path=os.path.join(FIG_DIR, "roc_A_val_multiseed.png"),
    title="ROC: A validation (multiseed)"
)

# 3) Multiseed PR on shared validation
mean_pr_curve(
    pr_items,
    fig_path=os.path.join(FIG_DIR, "pr_A_val_multiseed.png"),
    title="PR: A validation (multiseed)"
)

# 4) Multiseed Chebyshev degree sweep
plot_multiseed_degree_sweep(out_dir=OUT_DIR, fig_dir=FIG_DIR)

print("✅ Paper-ready multiseed figures created in:", FIG_DIR)

# Aggregate tables: Validation (A & B at t*), B_test pre/post
def agg_mean_std(rows: List[float]) -> Dict[str, float]:
    arr = np.array(rows, dtype=float)
    return dict(mean=float(arr.mean()), std=float(arr.std(ddof=0)))

def collect_metric(path_fn):
    vals = []
    for r in all_runs:
        try:
            v = path_fn(r)
        except KeyError:
            continue  # skip seeds missing this metric
        vals.append(v)
    return agg_mean_std(vals) if vals else dict(mean=float("nan"), std=float("nan"))


summary = {
    "seeds": [r["seed"] for r in all_runs],
    "A_val_AUROC": collect_metric(lambda r: r["val"]["A_at_tstar"]["AUROC"]),
    "A_val_AUPRC": collect_metric(lambda r: r["val"]["A_at_tstar"]["AUPRC"]),
    "A_val_F1":    collect_metric(lambda r: r["val"]["A_at_tstar"]["F1"]),
    "B_val_AUROC": collect_metric(lambda r: r["val"]["B_at_tstar"]["AUROC"]),
    "B_val_AUPRC": collect_metric(lambda r: r["val"]["B_at_tstar"]["AUPRC"]),
    "B_val_F1":    collect_metric(lambda r: r["val"]["B_at_tstar"]["F1"]),
    "Btest_pre_AUROC":  collect_metric(lambda r: r["btest_calibration"]["pre"]["AUROC"]),
    "Btest_pre_AUPRC":  collect_metric(lambda r: r["btest_calibration"]["pre"]["AUPRC"]),
    "Btest_pre_F1":     collect_metric(lambda r: r["btest_calibration"]["pre"]["F1"]),
    "Btest_pre_Brier":  collect_metric(lambda r: r["btest_calibration"]["pre"]["Brier"]),
    "Btest_pre_ECE":    collect_metric(lambda r: r["btest_calibration"]["pre"]["ECE"]),
    "Btest_post_AUROC": collect_metric(lambda r: r["btest_calibration"]["post"]["AUROC"]),
    "Btest_post_AUPRC": collect_metric(lambda r: r["btest_calibration"]["post"]["AUPRC"]),
    "Btest_post_F1":    collect_metric(lambda r: r["btest_calibration"]["post"]["F1"]),
    "Btest_post_Brier": collect_metric(lambda r: r["btest_calibration"]["post"]["Brier"]),
    "Btest_post_ECE":   collect_metric(lambda r: r["btest_calibration"]["post"]["ECE"]),
    "Calib_batch_ms":   collect_metric(lambda r: r["btest_calibration"]["batch_ms"]),
    "Btest_postTS_AUROC": collect_metric(lambda r: r["btest_calibration"]["post_TS"]["AUROC"]),
    "Btest_postTS_AUPRC": collect_metric(lambda r: r["btest_calibration"]["post_TS"]["AUPRC"]),
    "Btest_postTS_F1":    collect_metric(lambda r: r["btest_calibration"]["post_TS"]["F1"]),
    "Btest_postTS_Brier": collect_metric(lambda r: r["btest_calibration"]["post_TS"]["Brier"]),
    "Btest_postTS_ECE":   collect_metric(lambda r: r["btest_calibration"]["post_TS"]["ECE"]),

}

with open(os.path.join(OUT_DIR, "multi_seed_summary.json"), "w") as f:
    json.dump(summary, f, indent=2)

# Save per-seed baselines table
baseline_rows_all=[]
for r in all_runs:
    for b in r["baselines"]:
        baseline_rows_all.append(dict(seed=r["seed"], **b))
if baseline_rows_all:
    pd.DataFrame(baseline_rows_all).to_csv(os.path.join(OUT_DIR, "baseline_table_all_seeds.csv"), index=False)

# -----------------------------
# 9) Provenance & manifest dump
# -----------------------------
manifest = {
    "seeds": RUN_SEEDS,
    "platform": platform.platform(),
    "python": sys.version.split()[0],
    "versions": VERSIONS,
    "data_path": os.path.abspath(DATA_PATH),
    "n_docs_used": int(len(texts)),
    "embed": embed_manifest,
    "head": {"hidden_dim": HIDDEN_DIM, "dropout": 0.10, "num_classes": 2},
    "train": {"epochs": EPOCHS, "batch_size": BATCH_TRAIN, "lr": LR, "early_stop_patience": EARLY_STOP_PATIENCE},
    "partition": {"VAL_CAP": VAL_CAP, "PRIMARY_RATIO": PRIMARY_RATIO},
    "chebyshev": {"default_coeffs": CHEB_COEFFS, "degree_sweep": CHEB_DEGREES},
    "calibration": {"steps": CALIB_STEPS, "lr": CALIB_LR, "k_batch": K_CALIB, 
                "bias_only": bool(CALIB_BIAS_ONLY), "policy": CALIB_POLICY},
    "ckks_params_logged": CKKS_PARAMS,
    "outputs": {
        "multi_seed_summary": os.path.abspath(os.path.join(OUT_DIR, "multi_seed_summary.json")),
        "baseline_table_all_seeds": os.path.abspath(os.path.join(OUT_DIR, "baseline_table_all_seeds.csv")),
        "figs_dir": os.path.abspath(FIG_DIR)
    }
}
with open(os.path.join(OUT_DIR, "manifest.json"), "w") as f:
    json.dump(manifest, f, indent=2)

# ==========================
# 10) SUMMARIZERS (Code 2 + extras) — runs automatically
# ==========================
ROOT = OUT_DIR
JSON_PATH = os.path.join(ROOT, "multi_seed_summary.json")
OUT_LONG   = os.path.join(ROOT, "summary_long.csv")    # metric,mean,std
OUT_PRETTY = os.path.join(ROOT, "summary_pretty.csv")  # metric,mean_pm_std (formatted)

# ---- Code 2: summarize multi_seed_summary.json ----
with open(JSON_PATH, "r") as f:
    js = json.load(f)

rows = []
for k, v in js.items():
    if isinstance(v, dict) and {"mean","std"}.issubset(v):
        rows.append({"metric": k, "mean": v["mean"], "std": v["std"]})

df = pd.DataFrame(rows).sort_values("metric")

rename_map = {
  "A_val_AUROC":"Val (A) AUROC","A_val_AUPRC":"Val (A) AUPRC","A_val_F1":"Val (A) F1",
  "B_val_AUROC":"Val (B) AUROC","B_val_AUPRC":"Val (B) AUPRC","B_val_F1":"Val (B) F1",
  "Btest_pre_AUROC":"B_test Pre AUROC","Btest_pre_AUPRC":"B_test Pre AUPRC",
  "Btest_pre_F1":"B_test Pre F1","Btest_pre_Brier":"B_test Pre Brier","Btest_pre_ECE":"B_test Pre ECE",
  "Btest_post_AUROC":"B_test Post AUROC","Btest_post_AUPRC":"B_test Post AUPRC",
  "Btest_post_F1":"B_test Post F1","Btest_post_Brier":"B_test Post Brier","Btest_post_ECE":"B_test Post ECE",
  "Calib_batch_ms":"Calibration Batch Time (ms)",
  "Btest_postTS_AUROC":"B_test Post+TS AUROC",
  "Btest_postTS_AUPRC":"B_test Post+TS AUPRC",
  "Btest_postTS_F1":"B_test Post+TS F1",
  "Btest_postTS_Brier":"B_test Post+TS Brier",
  "Btest_postTS_ECE":"B_test Post+TS ECE",

}
df["metric"] = df["metric"].replace(rename_map)

# Save tidy (numeric) CSV
df.to_csv(OUT_LONG, index=False)

# Save pretty CSV with "mean ± std" text
def fmt(m, s, p=3): return f"{m:.{p}f} ± {s:.{p}f}"
df_pretty = df.assign(mean_pm_std=[fmt(m,s,3) for m,s in zip(df["mean"], df["std"])])[["metric","mean_pm_std"]]
df_pretty.to_csv(OUT_PRETTY, index=False)

# ---- Extras: per-seed calibration and degree sweep summaries ----
# 1) Per-seed calibration JSONs -> one table
rows = []
for path in sorted(glob.glob(os.path.join(ROOT, "Btest_calibration_seed*.json"))):
    try:
        seed = int(path.split("seed")[-1].split(".json")[0])
    except Exception:
        continue
    with open(path, "r") as f:
        js2 = json.load(f)
    pre  = js2.get("pre", {})
    post = js2.get("post", {})
    post_ts = js2.get("post_TS", {})

    rows.append({
        "seed": seed,
        "threshold_t*": pre.get("threshold", None),
        "pre_AUROC":  pre.get("AUROC", None),
        "pre_AUPRC":  pre.get("AUPRC", None),
        "pre_F1":     pre.get("F1", None),
        "pre_Brier":  pre.get("Brier", None),
        "pre_ECE":    pre.get("ECE", None),
        "post_AUROC": post.get("AUROC", None),
        "post_AUPRC": post.get("AUPRC", None),
        "post_F1":    post.get("F1", None),
        "post_Brier": post.get("Brier", None),
        "post_ECE":   post.get("ECE", None),
        "calib_batch_ms": js2.get("calibration_batch_ms", None),
        "k_used": js2.get("num_misclassified_used", js2.get("k_used", None)),
        "L_CHEB": js2.get("L_CHEB", None),
        "postTS_AUROC": post_ts.get("AUROC", None),
        "postTS_AUPRC": post_ts.get("AUPRC", None),
        "postTS_F1":    post_ts.get("F1", None),
        "postTS_Brier": post_ts.get("Brier", None),
        "postTS_ECE":   post_ts.get("ECE", None),
        "T_post":       post_ts.get("T", None),

    })

if rows:
    df_cal = pd.DataFrame(rows).sort_values("seed")
    df_cal.to_csv(os.path.join(ROOT, "per_seed_calibration_summary.csv"), index=False)

# 2) Degree sweep CSVs -> long table + mean±std
sweep_rows = []
for path in sorted(glob.glob(os.path.join(ROOT, "chebyshev_degree_sweep_seed*.csv"))):
    try:
        seed = int(path.split("seed")[-1].split(".csv")[0])
    except Exception:
        continue
    df_sw = pd.read_csv(path)
    df_sw["seed"] = seed
    sweep_rows.append(df_sw)

if sweep_rows:
    df_sweep_long = pd.concat(sweep_rows, ignore_index=True)
    df_sweep_long.to_csv(os.path.join(ROOT, "degree_sweep_long.csv"), index=False)

    agg = (df_sweep_long
           .groupby("degree")
           .agg(AUROC_mean=("AUROC","mean"), AUROC_std=("AUROC","std"),
                AUPRC_mean=("AUPRC","mean"), AUPRC_std=("AUPRC","std"),
                F1_mean=("F1","mean"),       F1_std=("F1","std"),
                Acc_mean=("Acc","mean"),     Acc_std=("Acc","std"),
                ECE_mean=("ECE","mean"),     ECE_std=("ECE","std"),
                time_ms_mean=("time_ms","mean"), time_ms_std=("time_ms","std"))
           .reset_index())
    agg.to_csv(os.path.join(ROOT, "degree_sweep_aggregate.csv"), index=False)

print("\n✅ Done. Artifacts written to:", os.path.abspath(OUT_DIR))
print("   - multi_seed_summary.json")
print("   - summary_long.csv / summary_pretty.csv")
print("   - per_seed_calibration_summary.csv (per-seed)")
print("   - degree_sweep_long.csv / degree_sweep_aggregate.csv (if degree-sweep ran)")


########################

# ================================================
# Merged post-processing: policy table + calibration baselines
# Produces:
#   - data/mis_selection_policy.csv
#   - data/calibration_baselines.csv   (only if logits files are present)
# ================================================
import os, glob, json
import numpy as np
import pandas as pd

# --------------------------
# Setup
# --------------------------
OUT_DIR = "q1_ckks_outputs"
DATA_DIR = "data"
os.makedirs(DATA_DIR, exist_ok=True)

# --------------------------
# Part A: Aggregate "policy" results from per-seed JSONs
# --------------------------
rows = []

# 1) Primary source: policy-tagged JSONs (optional; present only if you added policy logging)
for path in glob.glob(os.path.join(OUT_DIR, "*_policy_seed*.json")):
    with open(path) as f:
        j = json.load(f)
    post = j.get("post", {})
    pol  = j.get("policy", None)
    if post and pol:
        rows.append({
            "policy": pol,
            "F1": post.get("F1"),
            "ECE": post.get("ECE"),
            "Brier": post.get("Brier"),
        })

# 2) Fallback: use your existing per-seed calibration JSONs and treat as a single policy
if not rows:
    for path in glob.glob(os.path.join(OUT_DIR, "Btest_calibration_seed*.json")):
        with open(path) as f:
            j = json.load(f)
        # new format
        post = j.get("post", {})
        # older nested format
        if not post:
            post = j.get("btest_calibration", {}).get("post", {})
        if post:
            rows.append({
                "policy": "SingleBatch",
                "F1": post.get("F1"),
                "ECE": post.get("ECE"),
                "Brier": post.get("Brier"),
            })

# Write policy table
if not rows:
    pd.DataFrame([], columns=["policy","F1","ECE","Brier"]).to_csv(
        os.path.join(DATA_DIR, "mis_selection_policy.csv"), index=False
    )
    print("[warn] No calibration JSONs found. Wrote empty data/mis_selection_policy.csv")
else:
    df = pd.DataFrame(rows)
    agg = df.groupby("policy").agg(["mean","std"])
    agg.columns = ['_'.join(col) for col in agg.columns]
    agg = agg.reset_index()

    def fmt(m, s): 
        return f"{(m if m is not None else np.nan):.3f}±{(s if s is not None else np.nan):.3f}"

    out = pd.DataFrame({
        "policy": agg["policy"],
        "F1":    [fmt(m,s) for m,s in zip(agg.get("F1_mean", pd.Series(dtype=float)),
                                          agg.get("F1_std",  pd.Series(dtype=float)))],
        "ECE":   [fmt(m,s) for m,s in zip(agg.get("ECE_mean", pd.Series(dtype=float)),
                                          agg.get("ECE_std",  pd.Series(dtype=float)))],
        "Brier": [fmt(m,s) for m,s in zip(agg.get("Brier_mean", pd.Series(dtype=float)),
                                          agg.get("Brier_std",  pd.Series(dtype=float)))],
    })
    out.to_csv(os.path.join(DATA_DIR, "mis_selection_policy.csv"), index=False)
    print("Wrote", os.path.join(DATA_DIR, "mis_selection_policy.csv"))

# --------------------------
# Part B: Calibration baselines (TS / Platt / Isotonic)
# Requires logits CSVs; will skip gracefully if missing
# --------------------------
# Expected CSVs (produce these from Code 1; see my earlier patch):
#   - q1_ckks_outputs/ahead_val_logits.csv    # columns: logit, y
#   - q1_ckks_outputs/ahead_btest_logits.csv  # columns: logit, y
# Optionally also:
#   - q1_ckks_outputs/val_lr_tfidf.csv,  test_lr_tfidf.csv
#   - q1_ckks_outputs/val_lr_bert.csv,    test_lr_bert.csv

# helpers
def sigmoid(x):
    # use numpy; no external deps needed
    return 1.0 / (1.0 + np.exp(-x))

def temperature_scale_fit(logits_val, y_val):
    # Simple line search for binary temperature scaling
    t_candidates = np.linspace(0.5, 5.0, 91)
    best_t, best_nll = 1.0, 1e9
    eps = 1e-9
    p_true = y_val.astype(float)
    for t in t_candidates:
        p = sigmoid(logits_val / t)
        nll = -(p_true*np.log(p+eps) + (1-p_true)*np.log(1-p+eps)).mean()
        if nll < best_nll:
            best_t, best_nll = t, nll
    return best_t

def ece_uniform_bins(y_true, p, n_bins=15):
    y_true = np.asarray(y_true).astype(int)
    p = np.asarray(p).astype(float)
    bins = np.linspace(0.0, 1.0, n_bins+1)
    inds = np.digitize(p, bins) - 1
    accs, confs, weights = [], [], []
    for b in range(n_bins):
        m = inds == b
        if not np.any(m): 
            continue
        confs.append(p[m].mean())
        accs.append((y_true[m] == (p[m] >= 0.5)).mean())
        weights.append(m.mean())
    if not weights:
        return np.nan
    accs, confs, weights = np.array(accs), np.array(confs), np.array(weights)
    return float(np.sum(weights * np.abs(accs - confs)))

def metrics_block(y_true, p):
    from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, brier_score_loss
    auroc = roc_auc_score(y_true, p)
    auprc = average_precision_score(y_true, p)
    f1    = f1_score(y_true, (p >= 0.5).astype(int))
    ece   = ece_uniform_bins(y_true, p, n_bins=15)
    brier = brier_score_loss(y_true, p)
    return dict(AUROC=auroc, AUPRC=auprc, F1=f1, ECE=ece, Brier=brier)

def run_calibration_set(name, val_path, test_path):
    """Return rows for TS / Platt / Isotonic given val/test CSVs with 'logit','y'."""
    try:
        from sklearn.linear_model import LogisticRegression
        from sklearn.isotonic import IsotonicRegression
    except Exception as e:
        print(f"[warn] Skipping {name} calibrations; sklearn not available: {e}")
        return []

    if not (os.path.exists(val_path) and os.path.exists(test_path)):
        print(f"[warn] Skipping {name}: missing {val_path} or {test_path}")
        return []

    v = pd.read_csv(val_path)
    t = pd.read_csv(test_path)
    lv, yv = v["logit"].values, v["y"].values.astype(int)
    lt, yt = t["logit"].values, t["y"].values.astype(int)

    out = []

    # Temperature Scaling (TS)
    T = temperature_scale_fit(lv, yv)
    p_ts = sigmoid(lt / T)
    m_ts = metrics_block(yt, p_ts)
    out.append(dict(model=f"{name}+TS", **m_ts))

    # Platt scaling (LogisticRegression on logits)
    try:
        lr = LogisticRegression(solver="lbfgs", max_iter=1000)
        lr.fit(lv.reshape(-1,1), yv)
        p_platt = lr.predict_proba(lt.reshape(-1,1))[:,1]
        m_platt = metrics_block(yt, p_platt)
        out.append(dict(model=f"{name}+Platt", **m_platt))
    except Exception as e:
        print(f"[warn] Platt scaling failed for {name}: {e}")

    # Isotonic Regression
    try:
        iso = IsotonicRegression(out_of_bounds="clip")
        iso.fit(lv, yv)
        p_iso = iso.predict(lt)
        m_iso = metrics_block(yt, p_iso)
        out.append(dict(model=f"{name}+Isotonic", **m_iso))
    except Exception as e:
        print(f"[warn] Isotonic regression failed for {name}: {e}")

    return out

cal_rows = []
# A-head (export these from Code 1 per my earlier patch)
cal_rows += run_calibration_set(
    "A-head",
    os.path.join(OUT_DIR, "ahead_val_logits.csv"),
    os.path.join(OUT_DIR, "ahead_btest_logits.csv")
)

# Optional: LR(TF-IDF) and LR(BERT-emb) if you saved these logits
cal_rows += run_calibration_set(
    "LR(TF-IDF)",
    os.path.join(OUT_DIR, "val_lr_tfidf.csv"),
    os.path.join(OUT_DIR, "test_lr_tfidf.csv")
)
cal_rows += run_calibration_set(
    "LR(BERT-emb)",
    os.path.join(OUT_DIR, "val_lr_bert.csv"),
    os.path.join(OUT_DIR, "test_lr_bert.csv")
)

if cal_rows:
    df_cal = pd.DataFrame(cal_rows)
    df_cal.to_csv(os.path.join(DATA_DIR, "calibration_baselines.csv"), index=False)
    print("Wrote", os.path.join(DATA_DIR, "calibration_baselines.csv"))
else:
    # Always emit a file (empty) so your paper scripts don't break
    pd.DataFrame([], columns=["model","AUROC","AUPRC","F1","ECE","Brier"]).to_csv(
        os.path.join(DATA_DIR, "calibration_baselines.csv"), index=False
    )
    print("[warn] No logits CSVs found; wrote empty data/calibration_baselines.csv")

print("\n✅ Done.")


Python: 3.10.14
Platform: Linux-4.18.0-553.64.1.el8_10.x86_64-x86_64-with-glibc2.34
transformers: 4.40.1
torch: 2.0.1+cu117
tenseal: 0.3.16
sklearn: 1.6.1
pandas: 2.3.3
numpy: 1.26.4
matplotlib: 3.10.1
Dataset size: 1055348
Embeddings shape: (50000, 768)

============ SEED 42 ============
A train: (7335, 768) | B_train: (30931, 768) | B_calib: (3866, 768) | B_test: (3867, 768) | Val: (10000, 768)
Epoch 01: train loss=0.6790, val loss=0.6681
Epoch 02: train loss=0.6549, val loss=0.6602
Epoch 03: train loss=0.6420, val loss=0.6495
Epoch 04: train loss=0.6343, val loss=0.6499
Epoch 05: train loss=0.6326, val loss=0.6463
Epoch 06: train loss=0.6246, val loss=0.6453
Epoch 07: train loss=0.6206, val loss=0.6424
Epoch 08: train loss=0.6144, val loss=0.6457
Epoch 09: train loss=0.6060, val loss=0.6513
Epoch 10: train loss=0.6058, val loss=0.6508
Early stopping triggered at epoch 10.
Epoch 01: train loss=0.6642, val loss=0.6430
Epoch 02: train loss=0.6380, val loss=0.6347
Epoch 03: train loss=0